<a href="https://colab.research.google.com/github/MacuxiDev/sistema_copa_2026/blob/main/SistemaCopa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install sqlalchemy pydantic requests --quiet

In [2]:
import sqlalchemy
import pydantic
print(f"SQLAlchemy: {sqlalchemy.__version__}")
print(f"Pydantic: {pydantic.__version__}")

SQLAlchemy: 2.0.51
Pydantic: 2.13.4


In [3]:
# Célula 2 — Configuração
from sqlalchemy import create_engine
from sqlalchemy.orm import DeclarativeBase, Session

# Banco local no Colab
DATABASE_URL = "sqlite:///apostas_copa2026.db"
engine = create_engine(DATABASE_URL, echo=True)

class Base(DeclarativeBase):
    pass

print("✅ Engine configurada com sucesso")

✅ Engine configurada com sucesso


In [4]:
# Célula 3 — Models SQLAlchemy
from sqlalchemy import (
    Column, Integer, String, Float,
    DateTime, ForeignKey, CheckConstraint,
    UniqueConstraint, Text
)
from sqlalchemy.orm import relationship
from datetime import datetime, timezone

# ─────────────────────────────────────
# 1. FASE
# ─────────────────────────────────────
class Fase(Base):
    __tablename__ = "fase"

    id_fase  = Column(Integer, primary_key=True, autoincrement=True)
    nome     = Column(String(50), nullable=False, unique=True)
    ordem    = Column(Integer, nullable=False, unique=True)
    tipo     = Column(String(20), nullable=False)

    __table_args__ = (
        CheckConstraint("tipo IN ('grupos','eliminatoria')", name="ck_fase_tipo"),
        CheckConstraint("ordem > 0", name="ck_fase_ordem"),
    )

    grupos   = relationship("Grupo",   back_populates="fase")
    partidas = relationship("Partida", back_populates="fase")

    def __repr__(self):
        return f"<Fase {self.nome}>"


# ─────────────────────────────────────
# 2. GRUPO
# ─────────────────────────────────────
class Grupo(Base):
    __tablename__ = "grupo"

    id_grupo  = Column(Integer, primary_key=True, autoincrement=True)
    id_fase   = Column(Integer, ForeignKey("fase.id_fase"), nullable=False)
    nome      = Column(String(10), nullable=False)
    descricao = Column(String(100))

    __table_args__ = (
        UniqueConstraint("id_fase", "nome", name="uq_grupo_fase_nome"),
    )

    fase     = relationship("Fase",    back_populates="grupos")
    selecoes = relationship("Selecao", back_populates="grupo")

    def __repr__(self):
        return f"<Grupo {self.nome}>"


# ─────────────────────────────────────
# 3. SELECAO
# ─────────────────────────────────────
class Selecao(Base):
    __tablename__ = "selecao"

    id_selecao = Column(Integer, primary_key=True, autoincrement=True)
    id_grupo   = Column(Integer, ForeignKey("grupo.id_grupo"), nullable=False)
    nome       = Column(String(60), nullable=False)
    sigla      = Column(String(3),  nullable=False, unique=True)

    grupo = relationship("Grupo", back_populates="selecoes")

    partidas_a = relationship(
        "Partida",
        foreign_keys="Partida.id_selecao_a",
        back_populates="selecao_a"
    )
    partidas_b = relationship(
        "Partida",
        foreign_keys="Partida.id_selecao_b",
        back_populates="selecao_b"
    )

    def __repr__(self):
        return f"<Selecao {self.sigla}>"


# ─────────────────────────────────────
# 4. PARTIDA
# ─────────────────────────────────────
class Partida(Base):
    __tablename__ = "partida"

    id_partida   = Column(Integer, primary_key=True, autoincrement=True)
    id_fase      = Column(Integer, ForeignKey("fase.id_fase"),       nullable=False)
    id_selecao_a = Column(Integer, ForeignKey("selecao.id_selecao"), nullable=False)
    id_selecao_b = Column(Integer, ForeignKey("selecao.id_selecao"), nullable=False)
    num_jogo     = Column(Integer, nullable=False, unique=True)
    data_hora    = Column(DateTime)
    status       = Column(String(20), nullable=False, default="agendada")
    gols_a       = Column(Integer, default=0)
    gols_b       = Column(Integer, default=0)
    odd_a        = Column(Float, nullable=False)
    odd_b        = Column(Float, nullable=False)
    odd_empate   = Column(Float, nullable=False)

    __table_args__ = (
        CheckConstraint("id_selecao_a != id_selecao_b",           name="ck_partida_selecoes_diferentes"),
        CheckConstraint("status IN ('agendada','ao_vivo','encerrada','cancelada')", name="ck_partida_status"),
        CheckConstraint("odd_a > 0",      name="ck_partida_odd_a"),
        CheckConstraint("odd_b > 0",      name="ck_partida_odd_b"),
        CheckConstraint("odd_empate > 0", name="ck_partida_odd_empate"),
        CheckConstraint("gols_a >= 0",    name="ck_partida_gols_a"),
        CheckConstraint("gols_b >= 0",    name="ck_partida_gols_b"),
    )

    fase      = relationship("Fase",    back_populates="partidas")
    selecao_a = relationship("Selecao", foreign_keys=[id_selecao_a], back_populates="partidas_a")
    selecao_b = relationship("Selecao", foreign_keys=[id_selecao_b], back_populates="partidas_b")
    apostas   = relationship("Aposta",  back_populates="partida")

    def __repr__(self):
        return f"<Partida {self.num_jogo}>"


# ─────────────────────────────────────
# 5. USUARIO
# ─────────────────────────────────────
class Usuario(Base):
    __tablename__ = "usuario"

    id_usuario      = Column(Integer, primary_key=True, autoincrement=True)
    nome            = Column(String(100), nullable=False)
    email           = Column(String(120), nullable=False, unique=True)
    cpf             = Column(String(14),  nullable=False, unique=True)
    data_nascimento = Column(DateTime,    nullable=False)
    login           = Column(String(50),  nullable=False, unique=True)
    senha_hash      = Column(String(255), nullable=False)
    status          = Column(String(20),  nullable=False, default="ativo")
    tipo_usuario    = Column(String(20),  nullable=False, default="usuario")

    __table_args__ = (
        CheckConstraint("status IN ('ativo','inativo','bloqueado')",             name="ck_usuario_status"),
        CheckConstraint("tipo_usuario IN ('usuario','administrador')",           name="ck_usuario_tipo"),
    )

    conta   = relationship("Conta_Pontos", back_populates="usuario", uselist=False)
    apostas = relationship("Aposta",       back_populates="usuario")

    def __repr__(self):
        return f"<Usuario {self.login}>"


# ─────────────────────────────────────
# 6. CONTA_PONTOS
# ─────────────────────────────────────
class Conta_Pontos(Base):
    __tablename__ = "conta_pontos"

    id_conta        = Column(Integer, primary_key=True, autoincrement=True)
    id_usuario      = Column(Integer, ForeignKey("usuario.id_usuario"), nullable=False, unique=True)
    saldo           = Column(Float,   nullable=False, default=100.0)
    data_atualizacao = Column(DateTime, default=lambda: datetime.now(timezone.utc))

    __table_args__ = (
        CheckConstraint("saldo >= 0", name="ck_conta_saldo"),
    )

    usuario        = relationship("Usuario",              back_populates="conta")
    movimentacoes  = relationship("Movimentacao_Pontos",  back_populates="conta")

    def __repr__(self):
        return f"<Conta usuario={self.id_usuario} saldo={self.saldo}>"


# ─────────────────────────────────────
# 7. APOSTA
# ─────────────────────────────────────
class Aposta(Base):
    __tablename__ = "aposta"

    id_aposta        = Column(Integer, primary_key=True, autoincrement=True)
    id_usuario       = Column(Integer, ForeignKey("usuario.id_usuario"), nullable=False)
    id_partida       = Column(Integer, ForeignKey("partida.id_partida"), nullable=False)
    palpite          = Column(String(10), nullable=False)
    pontos_apostados = Column(Float,      nullable=False)
    multiplicador    = Column(Integer,    nullable=False, default=1)
    odd_aplicada     = Column(Float,      nullable=False)
    retorno_potencial = Column(Float,     nullable=False)
    status           = Column(String(20), nullable=False, default="ativa")
    data_hora        = Column(DateTime,   default=lambda: datetime.now(timezone.utc))

    __table_args__ = (
        CheckConstraint("palpite IN ('selecao_a','empate','selecao_b')", name="ck_aposta_palpite"),
        CheckConstraint("pontos_apostados > 0",                          name="ck_aposta_pontos"),
        CheckConstraint("multiplicador IN (1,2,3,4,5)",                  name="ck_aposta_multiplicador"),
        CheckConstraint("odd_aplicada > 0",                              name="ck_aposta_odd"),
        CheckConstraint("status IN ('ativa','ganha','perdida','reembolsada','anulada')", name="ck_aposta_status"),
    )

    usuario  = relationship("Usuario",  back_populates="apostas")
    partida  = relationship("Partida",  back_populates="apostas")
    movimentacoes = relationship("Movimentacao_Pontos", back_populates="aposta")

    def __repr__(self):
        return f"<Aposta {self.id_aposta} | {self.palpite} | {self.status}>"


# ─────────────────────────────────────
# 8. MOVIMENTACAO_PONTOS
# ─────────────────────────────────────
class Movimentacao_Pontos(Base):
    __tablename__ = "movimentacao_pontos"

    id_movimentacao   = Column(Integer, primary_key=True, autoincrement=True)
    id_conta          = Column(Integer, ForeignKey("conta_pontos.id_conta"), nullable=False)
    id_aposta         = Column(Integer, ForeignKey("aposta.id_aposta"),      nullable=True)
    tipo_movimentacao = Column(String(20), nullable=False)
    pontos            = Column(Float,      nullable=False)
    saldo_anterior    = Column(Float,      nullable=False)
    saldo_atual       = Column(Float,      nullable=False)
    data_hora         = Column(DateTime,   default=lambda: datetime.now(timezone.utc))
    descricao         = Column(Text)

    __table_args__ = (
        CheckConstraint(
            "tipo_movimentacao IN ('debito','credito','estorno','ajuste')",
            name="ck_mov_tipo"
        ),
    )

    conta  = relationship("Conta_Pontos", back_populates="movimentacoes")
    aposta = relationship("Aposta",       back_populates="movimentacoes")

    def __repr__(self):
        return f"<Mov {self.tipo_movimentacao} | {self.pontos}pts>"


# ─────────────────────────────────────
# CRIAR TODAS AS TABELAS
# ─────────────────────────────────────
Base.metadata.create_all(engine)
print("✅ Todas as tabelas criadas com sucesso!")

2026-07-14 21:55:57,966 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-14 21:55:57,968 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("fase")


INFO:sqlalchemy.engine.Engine:PRAGMA main.table_info("fase")


2026-07-14 21:55:57,969 INFO sqlalchemy.engine.Engine [raw sql] ()


...

2026-07-14 21:55:58,162 INFO sqlalchemy.engine.Engine [no key 0.00588s] ()


INFO:sqlalchemy.engine.Engine:[no key 0.00588s] ()


2026-07-14 21:55:58,178 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


✅ Todas as tabelas criadas com sucesso!


In [5]:
# Célula 4 — Seed OFICIAL: Fases, Grupos e Seleções Copa 2026
from sqlalchemy.orm import Session

with Session(engine) as session:

    # ─────────────────────────────────────
    # FASES
    # ─────────────────────────────────────
    fases = [
    Fase(nome="Fase de Grupos",    ordem=1, tipo="grupos"),
    Fase(nome="Rodada de 32",      ordem=2, tipo="eliminatoria"),  # ← NOVA
    Fase(nome="Oitavas de Final",  ordem=3, tipo="eliminatoria"),
    Fase(nome="Quartas de Final",  ordem=4, tipo="eliminatoria"),
    Fase(nome="Semifinal",         ordem=5, tipo="eliminatoria"),
    Fase(nome="Disputa 3o Lugar",  ordem=6, tipo="eliminatoria"),
    Fase(nome="Final",             ordem=7, tipo="eliminatoria"),
    ]
    session.add_all(fases)
    session.flush()

    fase_grupos = fases[0]

    # ─────────────────────────────────────
    # GRUPOS A → L  (12 grupos Copa 2026)
    # ─────────────────────────────────────
    letras = ["A","B","C","D","E","F","G","H","I","J","K","L"]
    grupos = [
        Grupo(id_fase=fase_grupos.id_fase, nome=f"Grupo {l}")
        for l in letras
    ]
    session.add_all(grupos)
    session.flush()

    grupo_map = {g.nome.split()[-1]: g for g in grupos}

    # ─────────────────────────────────────
    # SELEÇÕES OFICIAIS Copa 2026
    # ─────────────────────────────────────
    selecoes_dados = {
        "A": [
            ("México",            "MEX"),
            ("África do Sul",     "RSA"),
            ("Coreia do Sul",     "KOR"),
            ("República Tcheca",  "CZE"),
        ],
        "B": [
            ("Suíça",   "SUI"),
            ("Canadá",  "CAN"),
            ("Bósnia",  "BIH"),
            ("Catar",   "QAT"),
        ],
        "C": [
            ("Brasil",   "BRA"),
            ("Marrocos", "MAR"),
            ("Escócia",  "SCO"),
            ("Haiti",    "HAI"),
        ],
        "D": [
            ("Estados Unidos", "USA"),
            ("Austrália",      "AUS"),
            ("Paraguai",       "PAR"),
            ("Turquia",        "TUR"),
        ],
        "E": [
            ("Alemanha",       "GER"),
            ("Curaçau",        "CUW"),
            ("Costa do Marfim","CIV"),
            ("Equador",        "ECU"),
        ],
        "F": [
            ("Holanda", "NED"),
            ("Japão",   "JPN"),
            ("Tunísia", "TUN"),
            ("Suécia",  "SWE"),
        ],
        "G": [
            ("Bélgica",       "BEL"),
            ("Egito",         "EGY"),
            ("Irã",           "IRN"),
            ("Nova Zelândia", "NZL"),
        ],
        "H": [
            ("Espanha",        "ESP"),
            ("Cabo Verde",     "CPV"),
            ("Arábia Saudita", "KSA"),
            ("Uruguai",        "URU"),
        ],
        "I": [
            ("França",   "FRA"),
            ("Senegal",  "SEN"),
            ("Iraque",   "IRQ"),
            ("Noruega",  "NOR"),
        ],
        "J": [
            ("Argentina", "ARG"),
            ("Argélia",   "ALG"),
            ("Áustria",   "AUT"),
            ("Jordânia",  "JOR"),
        ],
        "K": [
            ("Portugal",             "POR"),
            ("Rep. Dem. do Congo",   "COD"),
            ("Uzbequistão",          "UZB"),
            ("Colômbia",             "COL"),
        ],
        "L": [
            ("Inglaterra", "ENG"),
            ("Croácia",    "CRO"),
            ("Gana",       "GHA"),
            ("Panamá",     "PAN"),
        ],
    }

    todas_selecoes = []
    for letra, times in selecoes_dados.items():
        g = grupo_map[letra]
        for nome, sigla in times:
            todas_selecoes.append(
                Selecao(id_grupo=g.id_grupo, nome=nome, sigla=sigla)
            )

    session.add_all(todas_selecoes)
    session.commit()

    # ─────────────────────────────────────
    # RESUMO
    # ─────────────────────────────────────
    print(f"✅ {len(fases)} fases inseridas")
    print(f"✅ {len(grupos)} grupos inseridos (A → L)")
    print(f"✅ {len(todas_selecoes)} seleções inseridas")
    print()
    print("─── Times por grupo ───")
    for letra, times in selecoes_dados.items():
        nomes = " | ".join(t[0] for t in times)
        print(f"  Grupo {letra}: {nomes}")

2026-07-14 21:55:58,240 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-14 21:55:58,244 INFO sqlalchemy.engine.Engine INSERT INTO fase (nome, ordem, tipo) VALUES (?, ?, ?) RETURNING id_fase


INFO:sqlalchemy.engine.Engine:INSERT INTO fase (nome, ordem, tipo) VALUES (?, ?, ?) RETURNING id_fase


2026-07-14 21:55:58,245 INFO sqlalchemy.engine.Engine [generated in 0.00016s (insertmanyvalues) 1/7 (ordered; batch not supported)] ('Fase de Grupos', 1, 'grupos')


...

2026-07-14 21:55:58,481 INFO sqlalchemy.engine.Engine [insertmanyvalues 48/48 (ordered; batch not supported)] (12, 'Panamá', 'PAN')


INFO:sqlalchemy.engine.Engine:[insertmanyvalues 48/48 (ordered; batch not supported)] (12, 'Panamá', 'PAN')


2026-07-14 21:55:58,484 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


✅ 7 fases inseridas
✅ 12 grupos inseridos (A → L)
✅ 48 seleções inseridas

─── Times por grupo ───
  Grupo A: México | África do Sul | Coreia do Sul | República Tcheca
  Grupo B: Suíça | Canadá | Bósnia | Catar
  Grupo C: Brasil | Marrocos | Escócia | Haiti
  Grupo D: Estados Unidos | Austrália | Paraguai | Turquia
  Grupo E: Alemanha | Curaçau | Costa do Marfim | Equador
  Grupo F: Holanda | Japão | Tunísia | Suécia
  Grupo G: Bélgica | Egito | Irã | Nova Zelândia
  Grupo H: Espanha | Cabo Verde | Arábia Saudita | Uruguai
  Grupo I: França | Senegal | Iraque | Noruega
  Grupo J: Argentina | Argélia | Áustria | Jordânia
  Grupo K: Portugal | Rep. Dem. do Congo | Uzbequistão | Colômbia
  Grupo L: Inglaterra | Croácia | Gana | Panamá


In [6]:
# Célula 5 — Seed: 72 Partidas da Fase de Grupos
from sqlalchemy.orm import Session
from itertools import combinations
from datetime import datetime, timedelta

with Session(engine) as session:

    # ─────────────────────────────────────
    # VERIFICAR SE JÁ EXISTEM PARTIDAS
    # ─────────────────────────────────────
    qtd_existente = session.query(Partida).count()
    if qtd_existente > 0:
        print(f"⚠️ Já existem {qtd_existente} partidas no banco.")
        print("   Limpe a tabela antes de rodar novamente.")
    else:
        # ─────────────────────────────────────
        # BUSCAR FASE DE GRUPOS
        # ─────────────────────────────────────
        fase_grupos = session.query(Fase).filter_by(nome="Fase de Grupos").one()

        # ─────────────────────────────────────
        # BUSCAR SELEÇÕES ORGANIZADAS POR GRUPO
        # ─────────────────────────────────────
        selecoes_por_grupo = {}

        rows = (
            session.query(Grupo, Selecao)
            .join(Selecao, Grupo.id_grupo == Selecao.id_grupo)
            .filter(Grupo.id_fase == fase_grupos.id_fase)
            .order_by(Grupo.nome, Selecao.id_selecao)
            .all()
        )

        for grupo, selecao in rows:
            selecoes_por_grupo.setdefault(grupo.nome, []).append(selecao)

        # ─────────────────────────────────────
        # GERAR 6 JOGOS POR GRUPO = 72 PARTIDAS
        # ─────────────────────────────────────
        partidas  = []
        num_jogo  = 1
        data_base = datetime(2026, 6, 11, 13, 0, 0)

        for grupo_nome in sorted(selecoes_por_grupo.keys()):
            selecoes = selecoes_por_grupo[grupo_nome]
            for selecao_a, selecao_b in combinations(selecoes, 2):
                partida = Partida(
                    id_fase      = fase_grupos.id_fase,
                    id_selecao_a = selecao_a.id_selecao,
                    id_selecao_b = selecao_b.id_selecao,
                    num_jogo     = num_jogo,
                    data_hora    = data_base + timedelta(hours=(num_jogo - 1) * 3),
                    status       = "agendada",
                    gols_a       = 0,
                    gols_b       = 0,
                    odd_a        = round(1.50 + ((num_jogo % 6) * 0.15), 2),
                    odd_b        = round(1.60 + ((num_jogo % 6) * 0.12), 2),
                    odd_empate   = round(3.00 + ((num_jogo % 4) * 0.20), 2),
                )
                partidas.append(partida)
                num_jogo += 1

        session.add_all(partidas)
        session.commit()

        # ─────────────────────────────────────
        # RESUMO
        # ─────────────────────────────────────
        print(f"✅ {len(partidas)} partidas inseridas com sucesso!")
        print(f"✅ Cobertura: {len(selecoes_por_grupo)} grupos × 6 jogos")
        print(f"✅ Numeração: Jogo 1 → Jogo {len(partidas)}")
        print(f"✅ Período:   11/06/2026 até {(data_base + timedelta(hours=(len(partidas)-1)*3)).strftime('%d/%m/%Y')}")
        print()
        print("─── Jogos por grupo ───")
        jogo_num = 1
        for grupo_nome in sorted(selecoes_por_grupo.keys()):
            selecoes = selecoes_por_grupo[grupo_nome]
            print(f"\n  {grupo_nome}:")
            for s_a, s_b in combinations(selecoes, 2):
                print(f"    Jogo {jogo_num:02d}: {s_a.nome} vs {s_b.nome}")
                jogo_num += 1

2026-07-14 21:55:58,555 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-14 21:55:58,566 INFO sqlalchemy.engine.Engine SELECT count(*) AS count_1 
FROM (SELECT partida.id_partida AS partida_id_partida, partida.id_fase AS partida_id_fase, partida.id_selecao_a AS partida_id_selecao_a, partida.id_selecao_b AS partida_id_selecao_b, partida.num_jogo AS partida_num_jogo, partida.data_hora AS partida_data_hora, partida.status AS partida_status, partida.gols_a AS partida_gols_a, partida.gols_b AS partida_gols_b, partida.odd_a AS partida_odd_a, partida.odd_b AS partida_odd_b, partida.odd_empate AS partida_odd_empate 
FROM partida) AS anon_1


INFO:sqlalchemy.engine.Engine:SELECT count(*) AS count_1 
FROM (SELECT partida.id_partida AS partida_id_partida, partida.id_fase AS partida_id_fase, partida.id_selecao_a AS partida_id_selecao_a, partida.id_selecao_b AS partida_id_selecao_b, partida.num_jogo AS partida_num_jogo, partida.data_hora AS partida_data_hora, partida.status AS partida_status, partida.gols_a AS partida_gols_a, partida.gols_b AS partida_gols_b, partida.odd_a AS partida_odd_a, partida.odd_b AS partida_odd_b, partida.odd_empate AS partida_odd_empate 
FROM partida) AS anon_1


2026-07-14 21:55:58,567 INFO sqlalchemy.engine.Engine [generated in 0.00163s] ()


...

INFO:sqlalchemy.engine.Engine:SELECT selecao.id_selecao AS selecao_id_selecao, selecao.id_grupo AS selecao_id_grupo, selecao.nome AS selecao_nome, selecao.sigla AS selecao_sigla 
FROM selecao 
WHERE selecao.id_selecao = ?


2026-07-14 21:55:58,927 INFO sqlalchemy.engine.Engine [cached since 0.1478s ago] (48,)


INFO:sqlalchemy.engine.Engine:[cached since 0.1478s ago] (48,)


    Jogo 69: Inglaterra vs Panamá
    Jogo 70: Croácia vs Gana
    Jogo 71: Croácia vs Panamá
    Jogo 72: Gana vs Panamá
2026-07-14 21:55:58,928 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK


In [7]:
# Célula 6 — Funções de Usuário
import hashlib
import re
from datetime import datetime, timezone
from sqlalchemy.orm import Session

# ─────────────────────────────────────────────────────
# HELPERS INTERNOS
# ─────────────────────────────────────────────────────

def _hash_senha(senha: str) -> str:
    """Gera hash SHA-256 da senha."""
    return hashlib.sha256(senha.encode()).hexdigest()


def _validar_senha(senha: str) -> tuple[bool, str]:
    """
    Regras:
    - Mínimo 8 caracteres
    - Pelo menos 1 letra maiúscula
    - Pelo menos 1 letra minúscula
    - Pelo menos 1 número
    - Pelo menos 1 caractere especial
    """
    if len(senha) < 8:
        return False, "Senha deve ter no mínimo 8 caracteres."
    if not re.search(r"[A-Z]", senha):
        return False, "Senha deve conter pelo menos 1 letra maiúscula."
    if not re.search(r"[a-z]", senha):
        return False, "Senha deve conter pelo menos 1 letra minúscula."
    if not re.search(r"\d", senha):
        return False, "Senha deve conter pelo menos 1 número."
    if not re.search(r"[!@#$%^&*(),.?\":{}|<>]", senha):
        return False, "Senha deve conter pelo menos 1 caractere especial."
    return True, "Senha válida."


def _validar_idade(data_nascimento: datetime) -> tuple[bool, str]:
    """Usuário deve ter pelo menos 18 anos."""
    hoje = datetime.now(timezone.utc).date()
    nascimento = data_nascimento.date() if hasattr(data_nascimento, 'date') else data_nascimento
    idade = (hoje - nascimento).days // 365
    if idade < 18:
        return False, f"Usuário possui {idade} anos. Idade mínima é 18 anos."
    return True, f"Idade válida ({idade} anos)."


def _validar_cpf(cpf: str) -> tuple[bool, str]:
    """Valida formato do CPF (000.000.000-00)."""
    cpf_limpo = re.sub(r"\D", "", cpf)
    if len(cpf_limpo) != 11:
        return False, "CPF inválido. Use o formato 000.000.000-00."
    if len(set(cpf_limpo)) == 1:
        return False, "CPF inválido (todos os dígitos iguais)."
    return True, "CPF válido."


def _validar_email(email: str) -> tuple[bool, str]:
    """Valida formato básico de e-mail."""
    padrao = r"^[\w\.-]+@[\w\.-]+\.\w{2,}$"
    if not re.match(padrao, email):
        return False, "E-mail inválido."
    return True, "E-mail válido."


# ─────────────────────────────────────────────────────
# FUNÇÃO 1 — CRIAR USUÁRIO
# ─────────────────────────────────────────────────────

def criar_usuario(
    nome: str,
    email: str,
    cpf: str,
    data_nascimento: datetime,
    login: str,
    senha: str,
    tipo_usuario: str = "usuario"
) -> dict:
    """
    Cria um novo usuário com conta de pontos (saldo inicial = 100).
    Retorna dict com sucesso/erro e dados do usuário criado.
    """

    # Validações
    ok, msg = _validar_email(email)
    if not ok:
        return {"sucesso": False, "erro": msg}

    ok, msg = _validar_cpf(cpf)
    if not ok:
        return {"sucesso": False, "erro": msg}

    ok, msg = _validar_idade(data_nascimento)
    if not ok:
        return {"sucesso": False, "erro": msg}

    ok, msg = _validar_senha(senha)
    if not ok:
        return {"sucesso": False, "erro": msg}

    if tipo_usuario not in ("usuario", "administrador"):
        return {"sucesso": False, "erro": "Tipo de usuário inválido."}

    with Session(engine) as session:
        # Verificar duplicidade
        if session.query(Usuario).filter_by(email=email).first():
            return {"sucesso": False, "erro": f"E-mail '{email}' já cadastrado."}
        if session.query(Usuario).filter_by(cpf=cpf).first():
            return {"sucesso": False, "erro": f"CPF '{cpf}' já cadastrado."}
        if session.query(Usuario).filter_by(login=login).first():
            return {"sucesso": False, "erro": f"Login '{login}' já cadastrado."}

        # Criar usuário
        usuario = Usuario(
            nome=nome,
            email=email,
            cpf=cpf,
            data_nascimento=data_nascimento,
            login=login,
            senha_hash=_hash_senha(senha),
            status="ativo",
            tipo_usuario=tipo_usuario,
        )
        session.add(usuario)
        session.flush()

        # Criar conta de pontos (saldo inicial = 100)
        conta = Conta_Pontos(
            id_usuario=usuario.id_usuario,
            saldo=100.0,
            data_atualizacao=datetime.now(timezone.utc),
        )
        session.add(conta)
        session.flush()

        # Registrar movimentação inicial
        mov = Movimentacao_Pontos(
            id_conta=conta.id_conta,
            id_aposta=None,
            tipo_movimentacao="credito",
            pontos=100.0,
            saldo_anterior=0.0,
            saldo_atual=100.0,
            data_hora=datetime.now(timezone.utc),
            descricao="Bônus de boas-vindas",
        )
        session.add(mov)
        session.commit()

        return {
            "sucesso":      True,
            "id_usuario":   usuario.id_usuario,
            "id_conta":     conta.id_conta,
            "nome":         usuario.nome,
            "login":        usuario.login,
            "tipo_usuario": usuario.tipo_usuario,
            "saldo":        conta.saldo,
            "mensagem":     f"Usuário '{login}' criado com sucesso! Saldo inicial: 100 pontos.",
        }


# ─────────────────────────────────────────────────────
# FUNÇÃO 2 — LOGIN
# ─────────────────────────────────────────────────────

def login_usuario(login: str, senha: str) -> dict:
    """
    Autentica o usuário.
    Retorna dados do usuário e saldo se OK.
    """
    with Session(engine) as session:
        usuario = session.query(Usuario).filter_by(login=login).first()

        if not usuario:
            return {"sucesso": False, "erro": "Login não encontrado."}

        if usuario.status == "inativo":
            return {"sucesso": False, "erro": "Conta cancelada. Acesso negado."}

        if usuario.status == "bloqueado":
            return {"sucesso": False, "erro": "Conta bloqueada. Entre em contato com o suporte."}

        if usuario.senha_hash != _hash_senha(senha):
            return {"sucesso": False, "erro": "Senha incorreta."}

        saldo = usuario.conta.saldo if usuario.conta else 0.0

        return {
            "sucesso":      True,
            "id_usuario":   usuario.id_usuario,
            "nome":         usuario.nome,
            "login":        usuario.login,
            "tipo_usuario": usuario.tipo_usuario,
            "status":       usuario.status,
            "saldo":        saldo,
            "mensagem":     f"Bem-vindo, {usuario.nome}!",
        }


# ─────────────────────────────────────────────────────
# FUNÇÃO 3 — TROCAR SENHA
# ─────────────────────────────────────────────────────

def trocar_senha(login: str, senha_atual: str, nova_senha: str) -> dict:
    """Troca a senha do usuário após validar a senha atual."""
    with Session(engine) as session:
        usuario = session.query(Usuario).filter_by(login=login).first()

        if not usuario:
            return {"sucesso": False, "erro": "Usuário não encontrado."}

        if usuario.senha_hash != _hash_senha(senha_atual):
            return {"sucesso": False, "erro": "Senha atual incorreta."}

        ok, msg = _validar_senha(nova_senha)
        if not ok:
            return {"sucesso": False, "erro": msg}

        if senha_atual == nova_senha:
            return {"sucesso": False, "erro": "Nova senha não pode ser igual à senha atual."}

        usuario.senha_hash = _hash_senha(nova_senha)
        session.commit()

        return {"sucesso": True, "mensagem": "Senha alterada com sucesso."}


# ─────────────────────────────────────────────────────
# FUNÇÃO 4 — CANCELAR USUÁRIO (soft delete)
# ─────────────────────────────────────────────────────

def cancelar_usuario(login: str, senha: str) -> dict:
    """
    Cancela o usuário (status = 'inativo').
    Não deleta fisicamente — decisão HU9/HU5.
    """
    with Session(engine) as session:
        usuario = session.query(Usuario).filter_by(login=login).first()

        if not usuario:
            return {"sucesso": False, "erro": "Usuário não encontrado."}

        if usuario.senha_hash != _hash_senha(senha):
            return {"sucesso": False, "erro": "Senha incorreta."}

        if usuario.status == "inativo":
            return {"sucesso": False, "erro": "Conta já está cancelada."}

        usuario.status = "inativo"
        session.commit()

        return {"sucesso": True, "mensagem": f"Conta '{login}' cancelada com sucesso."}


# ─────────────────────────────────────────────────────
# FUNÇÃO 5 — BLOQUEAR USUÁRIO (admin ou sistema)
# ─────────────────────────────────────────────────────

def bloquear_usuario(login: str, motivo: str = "Saldo zerado") -> dict:
    """Bloqueia um usuário (status = 'bloqueado')."""
    with Session(engine) as session:
        usuario = session.query(Usuario).filter_by(login=login).first()

        if not usuario:
            return {"sucesso": False, "erro": "Usuário não encontrado."}

        if usuario.status == "bloqueado":
            return {"sucesso": False, "erro": "Usuário já está bloqueado."}

        usuario.status = "bloqueado"
        session.commit()

        return {
            "sucesso":  True,
            "mensagem": f"Usuário '{login}' bloqueado. Motivo: {motivo}",
        }


# ─────────────────────────────────────────────────────
# FUNÇÃO 6 — CONSULTAR USUÁRIO
# ─────────────────────────────────────────────────────

def consultar_usuario(login: str) -> dict:
    """Retorna dados do usuário pelo login."""
    with Session(engine) as session:
        usuario = session.query(Usuario).filter_by(login=login).first()

        if not usuario:
            return {"sucesso": False, "erro": "Usuário não encontrado."}

        saldo = usuario.conta.saldo if usuario.conta else 0.0

        return {
            "sucesso":          True,
            "id_usuario":       usuario.id_usuario,
            "nome":             usuario.nome,
            "email":            usuario.email,
            "cpf":              usuario.cpf,
            "login":            usuario.login,
            "status":           usuario.status,
            "tipo_usuario":     usuario.tipo_usuario,
            "data_nascimento":  usuario.data_nascimento.strftime("%d/%m/%Y"),
            "saldo":            saldo,
        }


print("✅ Célula 6 carregada — 6 funções de usuário disponíveis:")
print("   → criar_usuario()")
print("   → login_usuario()")
print("   → trocar_senha()")
print("   → cancelar_usuario()")
print("   → bloquear_usuario()")
print("   → consultar_usuario()")

✅ Célula 6 carregada — 6 funções de usuário disponíveis:
   → criar_usuario()
   → login_usuario()
   → trocar_senha()
   → cancelar_usuario()
   → bloquear_usuario()
   → consultar_usuario()


In [8]:
# Célula 6a — Testes das funções de usuário
print("=" * 55)
print("TESTES — FUNÇÕES DE USUÁRIO")
print("=" * 55)

# ─────────────────────────────────────
# TESTE 1 — Criar usuário válido
# ─────────────────────────────────────
print("\n📋 TESTE 1 — Criar usuário válido")
resultado = criar_usuario(
    nome="Lucas Silva",
    email="lucas@email.com",
    cpf="123.456.789-09",
    data_nascimento=datetime(1995, 6, 15),
    login="lucas95",
    senha="Lucas@2026",
    tipo_usuario="usuario"
)
print(resultado)

# ─────────────────────────────────────
# TESTE 2 — Login correto
# ─────────────────────────────────────
print("\n📋 TESTE 2 — Login correto")
resultado = login_usuario("lucas95", "Lucas@2026")
print(resultado)

# ─────────────────────────────────────
# TESTE 3 — Login com senha errada
# ─────────────────────────────────────
print("\n📋 TESTE 3 — Login com senha errada")
resultado = login_usuario("lucas95", "senhaerrada")
print(resultado)

# ─────────────────────────────────────
# TESTE 4 — Criar usuário menor de idade
# ─────────────────────────────────────
print("\n📋 TESTE 4 — Criar usuário menor de idade")
resultado = criar_usuario(
    nome="Jovem Silva",
    email="jovem@email.com",
    cpf="111.222.333-44",
    data_nascimento=datetime(2012, 1, 1),
    login="jovem12",
    senha="Jovem@2026"
)
print(resultado)

# ─────────────────────────────────────
# TESTE 5 — Criar usuário senha fraca
# ─────────────────────────────────────
print("\n📋 TESTE 5 — Senha fraca")
resultado = criar_usuario(
    nome="Teste Fraco",
    email="fraco@email.com",
    cpf="999.888.777-66",
    data_nascimento=datetime(1990, 3, 20),
    login="testefraco",
    senha="abc123"
)
print(resultado)

# ─────────────────────────────────────
# TESTE 6 — Login duplicado
# ─────────────────────────────────────
print("\n📋 TESTE 6 — Login duplicado")
resultado = criar_usuario(
    nome="Lucas Cópia",
    email="outro@email.com",
    cpf="000.111.222-33",
    data_nascimento=datetime(1995, 6, 15),
    login="lucas95",
    senha="Lucas@2026"
)
print(resultado)

# ─────────────────────────────────────
# TESTE 7 — Consultar usuário
# ─────────────────────────────────────
print("\n📋 TESTE 7 — Consultar usuário")
resultado = consultar_usuario("lucas95")
print(resultado)

# ─────────────────────────────────────
# TESTE 8 — Trocar senha
# ─────────────────────────────────────
print("\n📋 TESTE 8 — Trocar senha")
resultado = trocar_senha("lucas95", "Lucas@2026", "Nova@Senha99")
print(resultado)

# ─────────────────────────────────────
# TESTE 9 — Cancelar usuário
# ─────────────────────────────────────
print("\n📋 TESTE 9 — Cancelar usuário")
resultado = cancelar_usuario("lucas95", "Nova@Senha99")
print(resultado)

# ─────────────────────────────────────
# TESTE 10 — Login após cancelamento
# ─────────────────────────────────────
print("\n📋 TESTE 10 — Login após cancelamento")
resultado = login_usuario("lucas95", "Nova@Senha99")
print(resultado)

print("\n" + "=" * 55)
print("TESTES CONCLUÍDOS")
print("=" * 55)

TESTES — FUNÇÕES DE USUÁRIO

📋 TESTE 1 — Criar usuário válido
2026-07-14 21:55:59,019 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-14 21:55:59,025 INFO sqlalchemy.engine.Engine SELECT usuario.id_usuario AS usuario_id_usuario, usuario.nome AS usuario_nome, usuario.email AS usuario_email, usuario.cpf AS usuario_cpf, usuario.data_nascimento AS usuario_data_nascimento, usuario.login AS usuario_login, usuario.senha_hash AS usuario_senha_hash, usuario.status AS usuario_status, usuario.tipo_usuario AS usuario_tipo_usuario 
FROM usuario 
WHERE usuario.email = ?
 LIMIT ? OFFSET ?


INFO:sqlalchemy.engine.Engine:SELECT usuario.id_usuario AS usuario_id_usuario, usuario.nome AS usuario_nome, usuario.email AS usuario_email, usuario.cpf AS usuario_cpf, usuario.data_nascimento AS usuario_data_nascimento, usuario.login AS usuario_login, usuario.senha_hash AS usuario_senha_hash, usuario.status AS usuario_status, usuario.tipo_usuario AS usuario_tipo_usuario 
FROM usuario 
WHERE usuario.email = ?
 LIMIT ? OFFSET ?


2026-07-14 21:55:59,028 INFO sqlalchemy.engine.Engine [generated in 0.00287s] ('lucas@email.com', 1, 0)


...

2026-07-14 21:55:59,298 INFO sqlalchemy.engine.Engine [cached since 0.2589s ago] ('lucas95', 1, 0)


INFO:sqlalchemy.engine.Engine:[cached since 0.2589s ago] ('lucas95', 1, 0)


2026-07-14 21:55:59,303 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK


{'sucesso': False, 'erro': 'Conta cancelada. Acesso negado.'}

TESTES CONCLUÍDOS


In [9]:
# Célula 7 — Funções de Conta e Pontos
from datetime import datetime, timezone
from sqlalchemy.orm import Session

# ─────────────────────────────────────────────────────
# HELPERS INTERNOS
# ─────────────────────────────────────────────────────

def _obter_usuario_conta(session, login: str):
    """
    Busca usuário e conta de pontos pelo login.
    Retorna (usuario, conta, erro_dict).
    """
    usuario = session.query(Usuario).filter_by(login=login).first()
    if not usuario:
        return None, None, {"sucesso": False, "erro": "Usuário não encontrado."}

    if usuario.status == "inativo":
        return None, None, {"sucesso": False, "erro": "Conta cancelada. Operação não permitida."}

    if usuario.status == "bloqueado":
        return None, None, {"sucesso": False, "erro": "Conta bloqueada. Operação não permitida."}

    conta = session.query(Conta_Pontos).filter_by(id_usuario=usuario.id_usuario).first()
    if not conta:
        return None, None, {"sucesso": False, "erro": "Conta de pontos não encontrada."}

    return usuario, conta, None


def _registrar_movimentacao(session, conta, tipo_movimentacao, pontos, saldo_anterior, saldo_atual, descricao, id_aposta=None):
    """
    Registra uma movimentação de pontos no histórico.
    """
    mov = Movimentacao_Pontos(
        id_conta=conta.id_conta,
        id_aposta=id_aposta,
        tipo_movimentacao=tipo_movimentacao,
        pontos=pontos,
        saldo_anterior=saldo_anterior,
        saldo_atual=saldo_atual,
        data_hora=datetime.now(timezone.utc),
        descricao=descricao,
    )
    session.add(mov)
    return mov


# ─────────────────────────────────────────────────────
# FUNÇÃO 1 — CONSULTAR SALDO
# ─────────────────────────────────────────────────────

def consultar_saldo(login: str) -> dict:
    """
    Retorna saldo atual da conta de pontos do usuário.
    """
    with Session(engine) as session:
        usuario, conta, erro = _obter_usuario_conta(session, login)
        if erro:
            return erro

        return {
            "sucesso": True,
            "id_usuario": usuario.id_usuario,
            "login": usuario.login,
            "nome": usuario.nome,
            "id_conta": conta.id_conta,
            "saldo": float(conta.saldo),
            "data_atualizacao": conta.data_atualizacao.strftime("%d/%m/%Y %H:%M:%S")
                if conta.data_atualizacao else None,
        }


# ─────────────────────────────────────────────────────
# FUNÇÃO 2 — CREDITAR PONTOS
# ─────────────────────────────────────────────────────

def creditar_pontos(login: str, pontos: float, descricao: str = "Crédito manual", id_aposta=None) -> dict:
    """
    Credita pontos na conta do usuário e registra movimentação.
    """
    if pontos <= 0:
        return {"sucesso": False, "erro": "O valor do crédito deve ser maior que zero."}

    with Session(engine) as session:
        usuario, conta, erro = _obter_usuario_conta(session, login)
        if erro:
            return erro

        saldo_anterior = float(conta.saldo)
        saldo_atual = round(saldo_anterior + float(pontos), 2)

        conta.saldo = saldo_atual
        conta.data_atualizacao = datetime.now(timezone.utc)

        _registrar_movimentacao(
            session=session,
            conta=conta,
            tipo_movimentacao="credito",
            pontos=float(pontos),
            saldo_anterior=saldo_anterior,
            saldo_atual=saldo_atual,
            descricao=descricao,
            id_aposta=id_aposta,
        )

        session.commit()

        return {
            "sucesso": True,
            "login": login,
            "saldo_anterior": saldo_anterior,
            "creditado": float(pontos),
            "saldo_atual": saldo_atual,
            "mensagem": f"{float(pontos)} pontos creditados com sucesso.",
        }


# ─────────────────────────────────────────────────────
# FUNÇÃO 3 — DEBITAR PONTOS
# ─────────────────────────────────────────────────────

def debitar_pontos(login: str, pontos: float, descricao: str = "Débito manual", id_aposta=None) -> dict:
    """
    Debita pontos da conta do usuário e registra movimentação.
    """
    if pontos <= 0:
        return {"sucesso": False, "erro": "O valor do débito deve ser maior que zero."}

    with Session(engine) as session:
        usuario, conta, erro = _obter_usuario_conta(session, login)
        if erro:
            return erro

        saldo_anterior = float(conta.saldo)
        pontos = float(pontos)

        if saldo_anterior < pontos:
            return {
                "sucesso": False,
                "erro": f"Saldo insuficiente. Saldo atual: {saldo_anterior:.2f}."
            }

        saldo_atual = round(saldo_anterior - pontos, 2)

        conta.saldo = saldo_atual
        conta.data_atualizacao = datetime.now(timezone.utc)

        _registrar_movimentacao(
            session=session,
            conta=conta,
            tipo_movimentacao="debito",
            pontos=pontos,
            saldo_anterior=saldo_anterior,
            saldo_atual=saldo_atual,
            descricao=descricao,
            id_aposta=id_aposta,
        )

        session.commit()

        return {
            "sucesso": True,
            "login": login,
            "saldo_anterior": saldo_anterior,
            "debitado": pontos,
            "saldo_atual": saldo_atual,
            "mensagem": f"{pontos} pontos debitados com sucesso.",
        }


# ─────────────────────────────────────────────────────
# FUNÇÃO 4 — LISTAR MOVIMENTAÇÕES
# ─────────────────────────────────────────────────────

def listar_movimentacoes(login: str, limite: int = 10) -> dict:
    """
    Lista o histórico recente de movimentações do usuário.
    """
    if limite <= 0:
        return {"sucesso": False, "erro": "O limite deve ser maior que zero."}

    with Session(engine) as session:
        usuario = session.query(Usuario).filter_by(login=login).first()
        if not usuario:
            return {"sucesso": False, "erro": "Usuário não encontrado."}

        conta = session.query(Conta_Pontos).filter_by(id_usuario=usuario.id_usuario).first()
        if not conta:
            return {"sucesso": False, "erro": "Conta de pontos não encontrada."}

        movs = (
            session.query(Movimentacao_Pontos)
            .filter_by(id_conta=conta.id_conta)
            .order_by(Movimentacao_Pontos.data_hora.desc())
            .limit(limite)
            .all()
        )

        historico = []
        for m in movs:
            historico.append({
                "id_movimentacao": m.id_movimentacao,
                "tipo_movimentacao": m.tipo_movimentacao,
                "pontos": float(m.pontos),
                "saldo_anterior": float(m.saldo_anterior),
                "saldo_atual": float(m.saldo_atual),
                "data_hora": m.data_hora.strftime("%d/%m/%Y %H:%M:%S") if m.data_hora else None,
                "descricao": m.descricao,
                "id_aposta": m.id_aposta,
            })

        return {
            "sucesso": True,
            "login": login,
            "nome": usuario.nome,
            "id_conta": conta.id_conta,
            "saldo_atual": float(conta.saldo),
            "total_movimentacoes": len(historico),
            "historico": historico,
        }


# ─────────────────────────────────────────────────────
# FUNÇÃO 5 — AJUSTAR SALDO (GENÉRICA)
# ─────────────────────────────────────────────────────

def ajustar_saldo(login: str, valor: float, descricao: str = "Ajuste manual", id_aposta=None) -> dict:
    """
    Ajusta saldo com valor positivo ou negativo.
    Positivo = crédito | Negativo = débito
    """
    if valor == 0:
        return {"sucesso": False, "erro": "O ajuste não pode ser zero."}

    if valor > 0:
        return creditar_pontos(login, valor, descricao=descricao, id_aposta=id_aposta)

    return debitar_pontos(login, abs(valor), descricao=descricao, id_aposta=id_aposta)


print("✅ Célula 7 carregada — funções de conta e pontos disponíveis:")
print("   → consultar_saldo()")
print("   → creditar_pontos()")
print("   → debitar_pontos()")
print("   → listar_movimentacoes()")
print("   → ajustar_saldo()")

✅ Célula 7 carregada — funções de conta e pontos disponíveis:
   → consultar_saldo()
   → creditar_pontos()
   → debitar_pontos()
   → listar_movimentacoes()
   → ajustar_saldo()


In [10]:
# Célula 7a — Testes das funções de conta e pontos
from sqlalchemy.orm import Session

print("=" * 60)
print("TESTES — FUNÇÕES DE CONTA E PONTOS")
print("=" * 60)

LOGIN_TESTE = "conta_teste_2026"
SENHA_TESTE = "Conta@2026"

# ─────────────────────────────────────
# PASSO 1 — Garantir usuário de teste ativo
# ─────────────────────────────────────
with Session(engine) as session:
    usuario = session.query(Usuario).filter_by(login=LOGIN_TESTE).first()

    if not usuario:
        resultado_criacao = criar_usuario(
            nome="Conta Teste 2026",
            email="conta.teste2026@email.com",
            cpf="987.654.321-00",
            data_nascimento=datetime(1990, 1, 15),
            login=LOGIN_TESTE,
            senha=SENHA_TESTE,
            tipo_usuario="usuario"
        )
        print("\n📋 CRIAÇÃO DO USUÁRIO DE TESTE")
        print(resultado_criacao)
    else:
        if usuario.status != "ativo":
            usuario.status = "ativo"
            session.commit()
            print(f"\n✅ Usuário '{LOGIN_TESTE}' reativado para os testes.")
        else:
            print(f"\n✅ Usuário '{LOGIN_TESTE}' já está ativo.")

# ─────────────────────────────────────
# TESTE 1 — Consultar saldo inicial
# ─────────────────────────────────────
print("\n📋 TESTE 1 — Consultar saldo inicial")
resultado = consultar_saldo(LOGIN_TESTE)
print(resultado)

# ─────────────────────────────────────
# TESTE 2 — Creditar pontos
# ─────────────────────────────────────
print("\n📋 TESTE 2 — Creditar 50 pontos")
resultado = creditar_pontos(
    LOGIN_TESTE,
    50,
    descricao="Crédito de teste da Célula 7"
)
print(resultado)

# ─────────────────────────────────────
# TESTE 3 — Debitar pontos
# ─────────────────────────────────────
print("\n📋 TESTE 3 — Debitar 30 pontos")
resultado = debitar_pontos(
    LOGIN_TESTE,
    30,
    descricao="Débito de teste da Célula 7"
)
print(resultado)

# ─────────────────────────────────────
# TESTE 4 — Consultar saldo após movimentações
# ─────────────────────────────────────
print("\n📋 TESTE 4 — Consultar saldo após crédito e débito")
resultado = consultar_saldo(LOGIN_TESTE)
print(resultado)

# ─────────────────────────────────────
# TESTE 5 — Listar movimentações
# ─────────────────────────────────────
print("\n📋 TESTE 5 — Listar movimentações recentes")
resultado = listar_movimentacoes(LOGIN_TESTE, limite=5)
print(resultado)

# ─────────────────────────────────────
# TESTE 6 — Ajustar saldo negativo
# ─────────────────────────────────────
print("\n📋 TESTE 6 — Ajustar saldo em -20 pontos")
resultado = ajustar_saldo(
    LOGIN_TESTE,
    -20,
    descricao="Ajuste negativo de teste da Célula 7"
)
print(resultado)

# ─────────────────────────────────────
# TESTE 7 — Tentativa de débito acima do saldo
# ─────────────────────────────────────
print("\n📋 TESTE 7 — Débito acima do saldo")
resultado = debitar_pontos(
    LOGIN_TESTE,
    1000,
    descricao="Teste de saldo insuficiente"
)
print(resultado)

# ─────────────────────────────────────
# TESTE 8 — Saldo final
# ─────────────────────────────────────
print("\n📋 TESTE 8 — Saldo final")
resultado = consultar_saldo(LOGIN_TESTE)
print(resultado)

print("\n" + "=" * 60)
print("TESTES CONCLUÍDOS")
print("=" * 60)

TESTES — FUNÇÕES DE CONTA E PONTOS
2026-07-14 21:55:59,419 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-14 21:55:59,424 INFO sqlalchemy.engine.Engine SELECT usuario.id_usuario AS usuario_id_usuario, usuario.nome AS usuario_nome, usuario.email AS usuario_email, usuario.cpf AS usuario_cpf, usuario.data_nascimento AS usuario_data_nascimento, usuario.login AS usuario_login, usuario.senha_hash AS usuario_senha_hash, usuario.status AS usuario_status, usuario.tipo_usuario AS usuario_tipo_usuario 
FROM usuario 
WHERE usuario.login = ?
 LIMIT ? OFFSET ?


INFO:sqlalchemy.engine.Engine:SELECT usuario.id_usuario AS usuario_id_usuario, usuario.nome AS usuario_nome, usuario.email AS usuario_email, usuario.cpf AS usuario_cpf, usuario.data_nascimento AS usuario_data_nascimento, usuario.login AS usuario_login, usuario.senha_hash AS usuario_senha_hash, usuario.status AS usuario_status, usuario.tipo_usuario AS usuario_tipo_usuario 
FROM usuario 
WHERE usuario.login = ?
 LIMIT ? OFFSET ?


2026-07-14 21:55:59,429 INFO sqlalchemy.engine.Engine [cached since 0.3903s ago] ('conta_teste_2026', 1, 0)


...

2026-07-14 21:55:59,845 INFO sqlalchemy.engine.Engine [cached since 0.2874s ago] (2, 1, 0)


INFO:sqlalchemy.engine.Engine:[cached since 0.2874s ago] (2, 1, 0)


2026-07-14 21:55:59,848 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK


{'sucesso': True, 'id_usuario': 2, 'login': 'conta_teste_2026', 'nome': 'Conta Teste 2026', 'id_conta': 2, 'saldo': 100.0, 'data_atualizacao': '14/07/2026 21:55:59'}

TESTES CONCLUÍDOS


In [11]:
# Célula 8 — Funções de Aposta (versão corrigida e alinhada com o DDL real)
#
# DDL confirmado:
#   palpite       IN ('selecao_a', 'empate', 'selecao_b')
#   multiplicador IN (1, 2, 3, 4, 5)  → inteiro
#   status        IN ('ativa', 'ganha', 'perdida', 'reembolsada', 'anulada')

import unicodedata
from datetime import datetime, timezone
from sqlalchemy.orm import Session
from sqlalchemy import select

# ─────────────────────────────────────────────────────
# HELPERS INTERNOS
# ─────────────────────────────────────────────────────

def _normalizar_previsao(previsao: str) -> str:
    """
    Converte entrada do usuário para os valores aceitos pela constraint:
        selecao_a | empate | selecao_b
    """
    if not previsao:
        return ""

    texto = unicodedata.normalize("NFKD", str(previsao))
    texto = texto.encode("ASCII", "ignore").decode("ASCII").strip().upper()

    mapa = {
        # Selecao A
        "A":          "selecao_a",
        "SELECAO_A":  "selecao_a",
        "SELECAO A":  "selecao_a",
        "CASA":       "selecao_a",
        "MANDANTE":   "selecao_a",
        "TIME A":     "selecao_a",
        "1":          "selecao_a",

        # Selecao B
        "B":          "selecao_b",
        "SELECAO_B":  "selecao_b",
        "SELECAO B":  "selecao_b",
        "FORA":       "selecao_b",
        "VISITANTE":  "selecao_b",
        "TIME B":     "selecao_b",
        "2":          "selecao_b",

        # Empate
        "EMPATE":     "empate",
        "X":          "empate",
        "E":          "empate",
        "0":          "empate",
    }
    return mapa.get(texto, "")


def _odd_da_partida(partida, palpite: str) -> float:
    """Retorna a odd correta conforme o palpite normalizado."""
    if palpite == "selecao_a":
        return float(partida.odd_a)
    if palpite == "selecao_b":
        return float(partida.odd_b)
    if palpite == "empate":
        return float(partida.odd_empate)
    raise ValueError(f"Palpite inválido para cálculo de odd: '{palpite}'")


def _label_palpite(palpite: str, nome_a: str, nome_b: str) -> str:
    """Retorna descrição legível do palpite."""
    if palpite == "selecao_a":
        return f"{nome_a} vence"
    if palpite == "selecao_b":
        return f"{nome_b} vence"
    if palpite == "empate":
        return "Empate"
    return palpite


# ─────────────────────────────────────────────────────
# FUNÇÃO 1 — CONSULTAR PARTIDA PARA APOSTA
# ─────────────────────────────────────────────────────

def consultar_partida_para_aposta(id_partida: int) -> dict:
    """Retorna dados da partida e odds disponíveis."""
    with Session(engine) as session:
        partida = session.get(Partida, id_partida)
        if not partida:
            return {"sucesso": False, "erro": "Partida não encontrada."}

        sel_a = session.get(Selecao, partida.id_selecao_a)
        sel_b = session.get(Selecao, partida.id_selecao_b)

        return {
            "sucesso":     True,
            "id_partida":  partida.id_partida,
            "num_jogo":    partida.num_jogo,
            "status":      partida.status,
            "data_hora":   partida.data_hora.strftime("%d/%m/%Y %H:%M:%S") if partida.data_hora else None,
            "time_a":      sel_a.nome if sel_a else str(partida.id_selecao_a),
            "time_b":      sel_b.nome if sel_b else str(partida.id_selecao_b),
            "odd_a":       float(partida.odd_a),
            "odd_b":       float(partida.odd_b),
            "odd_empate":  float(partida.odd_empate),
            "opcoes": {
                "selecao_a": f"{sel_a.nome if sel_a else 'Time A'} vence → odd {float(partida.odd_a)}",
                "empate":    f"Empate → odd {float(partida.odd_empate)}",
                "selecao_b": f"{sel_b.nome if sel_b else 'Time B'} vence → odd {float(partida.odd_b)}",
            }
        }


# ─────────────────────────────────────────────────────
# FUNÇÃO 2 — SIMULAR RETORNO POTENCIAL
# ─────────────────────────────────────────────────────

def simular_retorno_aposta(id_partida: int, valor: float, previsao_resultado: str) -> dict:
    """Calcula retorno potencial sem gravar nada no banco."""
    if valor <= 0:
        return {"sucesso": False, "erro": "O valor da aposta deve ser maior que zero."}

    palpite = _normalizar_previsao(previsao_resultado)
    if not palpite:
        return {"sucesso": False, "erro": f"Previsão inválida: '{previsao_resultado}'. Use A, B ou EMPATE."}

    with Session(engine) as session:
        partida = session.get(Partida, id_partida)
        if not partida:
            return {"sucesso": False, "erro": "Partida não encontrada."}

        odd = _odd_da_partida(partida, palpite)
        retorno_potencial = round(float(valor) * odd, 2)
        lucro_potencial   = round(retorno_potencial - float(valor), 2)

        sel_a = session.get(Selecao, partida.id_selecao_a)
        sel_b = session.get(Selecao, partida.id_selecao_b)

        return {
            "sucesso":           True,
            "id_partida":        id_partida,
            "partida":           f"{sel_a.nome if sel_a else '?'} x {sel_b.nome if sel_b else '?'}",
            "valor_apostado":    float(valor),
            "palpite":           palpite,
            "descricao_palpite": _label_palpite(palpite, sel_a.nome if sel_a else "A", sel_b.nome if sel_b else "B"),
            "odd_utilizada":     odd,
            "retorno_potencial": retorno_potencial,
            "lucro_potencial":   lucro_potencial,
        }


# ─────────────────────────────────────────────────────
# FUNÇÃO 3 — CRIAR APOSTA
# ─────────────────────────────────────────────────────

def criar_aposta(login: str, id_partida: int, valor: float, previsao_resultado: str) -> dict:
    """
    Cria uma aposta com:
    - palpite normalizado para o schema real
    - odd automática da partida
    - retorno potencial calculado
    - débito atômico do saldo
    - status inicial = 'ativa'
    - multiplicador = 1 (sem bônus)
    """
    if valor <= 0:
        return {"sucesso": False, "erro": "O valor da aposta deve ser maior que zero."}

    palpite = _normalizar_previsao(previsao_resultado)
    if not palpite:
        return {"sucesso": False, "erro": f"Previsão inválida: '{previsao_resultado}'. Use A, B ou EMPATE."}

    with Session(engine) as session:
        try:
            # Usuário
            usuario = session.query(Usuario).filter_by(login=login).first()
            if not usuario:
                return {"sucesso": False, "erro": "Usuário não encontrado."}
            if usuario.status == "inativo":
                return {"sucesso": False, "erro": "Conta cancelada. Operação não permitida."}
            if usuario.status == "bloqueado":
                return {"sucesso": False, "erro": "Conta bloqueada. Operação não permitida."}

            # Conta
            conta = session.query(Conta_Pontos).filter_by(id_usuario=usuario.id_usuario).first()
            if not conta:
                return {"sucesso": False, "erro": "Conta de pontos não encontrada."}

            # Partida
            partida = session.get(Partida, id_partida)
            if not partida:
                return {"sucesso": False, "erro": "Partida não encontrada."}

            status_partida = str(partida.status or "").strip().lower()
            if status_partida not in ("agendada", "aberta", "pendente"):
                return {"sucesso": False, "erro": f"Partida indisponível para aposta. Status: '{partida.status}'."}

            # Verificar duplicidade
            aposta_existente = (
                session.query(Aposta)
                .filter_by(id_usuario=usuario.id_usuario, id_partida=id_partida)
                .first()
            )
            if aposta_existente:
                return {"sucesso": False, "erro": "Já existe uma aposta registrada para este usuário nesta partida."}

            # Verificar saldo
            saldo_anterior = float(conta.saldo)
            if saldo_anterior < float(valor):
                return {"sucesso": False, "erro": f"Saldo insuficiente. Saldo atual: {saldo_anterior:.2f}."}

            # Calcular odd e retorno
            odd             = _odd_da_partida(partida, palpite)
            retorno_potencial = round(float(valor) * odd, 2)
            lucro_potencial   = round(retorno_potencial - float(valor), 2)
            saldo_atual       = round(saldo_anterior - float(valor), 2)

            # Debitar saldo
            conta.saldo = saldo_atual
            conta.data_atualizacao = datetime.now(timezone.utc)

            # Criar aposta alinhada ao DDL real
            aposta = Aposta(
                id_usuario        = usuario.id_usuario,
                id_partida        = id_partida,
                palpite           = palpite,           # selecao_a | empate | selecao_b
                pontos_apostados  = float(valor),
                multiplicador     = 1,                 # inteiro, sem bônus
                odd_aplicada      = odd,
                retorno_potencial = retorno_potencial,
                status            = "ativa",           # constraint aceita: ativa
                data_hora         = datetime.now(timezone.utc),
            )
            session.add(aposta)
            session.flush()

            # Selecoes para mensagem
            sel_a = session.get(Selecao, partida.id_selecao_a)
            sel_b = session.get(Selecao, partida.id_selecao_b)
            nome_a = sel_a.nome if sel_a else str(partida.id_selecao_a)
            nome_b = sel_b.nome if sel_b else str(partida.id_selecao_b)

            # Registrar movimentação
            mov = Movimentacao_Pontos(
                id_conta          = conta.id_conta,
                id_aposta         = aposta.id_aposta,
                tipo_movimentacao = "debito",
                pontos            = float(valor),
                saldo_anterior    = saldo_anterior,
                saldo_atual       = saldo_atual,
                data_hora         = datetime.now(timezone.utc),
                descricao         = (
                    f"Aposta Jogo {partida.num_jogo}: "
                    f"{nome_a} x {nome_b} | "
                    f"{_label_palpite(palpite, nome_a, nome_b)} | "
                    f"Retorno potencial: {retorno_potencial:.2f}"
                ),
            )
            session.add(mov)
            session.commit()

            return {
                "sucesso":           True,
                "id_aposta":         aposta.id_aposta,
                "id_partida":        id_partida,
                "num_jogo":          partida.num_jogo,
                "login":             login,
                "partida":           f"{nome_a} x {nome_b}",
                "palpite":           palpite,
                "descricao_palpite": _label_palpite(palpite, nome_a, nome_b),
                "valor_apostado":    float(valor),
                "odd_utilizada":     odd,
                "retorno_potencial": retorno_potencial,
                "lucro_potencial":   lucro_potencial,
                "multiplicador":     1,
                "status_aposta":     "ativa",
                "saldo_anterior":    saldo_anterior,
                "saldo_atual":       saldo_atual,
                "mensagem":          "Aposta registrada com sucesso.",
            }

        except Exception as e:
            session.rollback()
            return {"sucesso": False, "erro": f"Erro inesperado: {str(e)}"}


# ─────────────────────────────────────────────────────
# FUNÇÃO 4 — LISTAR APOSTAS DO USUÁRIO
# ─────────────────────────────────────────────────────

def listar_apostas_usuario(login: str, limite: int = 10) -> dict:
    """Lista as apostas mais recentes do usuário com detalhes legíveis."""
    if limite <= 0:
        return {"sucesso": False, "erro": "O limite deve ser maior que zero."}

    with Session(engine) as session:
        usuario = session.query(Usuario).filter_by(login=login).first()
        if not usuario:
            return {"sucesso": False, "erro": "Usuário não encontrado."}

        apostas = (
            session.query(Aposta)
            .filter_by(id_usuario=usuario.id_usuario)
            .order_by(Aposta.id_aposta.desc())
            .limit(limite)
            .all()
        )

        historico = []
        for a in apostas:
            partida = session.get(Partida, a.id_partida)
            sel_a   = session.get(Selecao, partida.id_selecao_a) if partida else None
            sel_b   = session.get(Selecao, partida.id_selecao_b) if partida else None
            nome_a  = sel_a.nome if sel_a else "?"
            nome_b  = sel_b.nome if sel_b else "?"

            historico.append({
                "id_aposta":         a.id_aposta,
                "id_partida":        a.id_partida,
                "num_jogo":          partida.num_jogo if partida else None,
                "partida":           f"{nome_a} x {nome_b}",
                "palpite":           a.palpite,
                "descricao_palpite": _label_palpite(a.palpite, nome_a, nome_b),
                "valor_apostado":    float(a.pontos_apostados),
                "odd_utilizada":     float(a.odd_aplicada),
                "multiplicador":     a.multiplicador,
                "retorno_potencial": float(a.retorno_potencial),
                "status":            a.status,
                "data_hora":         a.data_hora.strftime("%d/%m/%Y %H:%M:%S") if a.data_hora else None,
            })

        return {
            "sucesso":       True,
            "login":         login,
            "nome":          usuario.nome,
            "total_apostas": len(historico),
            "apostas":       historico,
        }


print("✅ Célula 8 reescrita e alinhada ao DDL real da tabela Aposta")
print()
print("─── Constraints respeitadas ───────────────────────────")
print("  palpite       → 'selecao_a' | 'empate' | 'selecao_b'")
print("  multiplicador → 1 (inteiro, sem bônus)")
print("  status        → 'ativa' ao criar")
print()
print("─── Funções disponíveis ───────────────────────────────")
print("  → consultar_partida_para_aposta(id_partida)")
print("  → simular_retorno_aposta(id_partida, valor, previsao)")
print("  → criar_aposta(login, id_partida, valor, previsao)")
print("  → listar_apostas_usuario(login, limite)")

✅ Célula 8 reescrita e alinhada ao DDL real da tabela Aposta

─── Constraints respeitadas ───────────────────────────
  palpite       → 'selecao_a' | 'empate' | 'selecao_b'
  multiplicador → 1 (inteiro, sem bônus)
  status        → 'ativa' ao criar

─── Funções disponíveis ───────────────────────────────
  → consultar_partida_para_aposta(id_partida)
  → simular_retorno_aposta(id_partida, valor, previsao)
  → criar_aposta(login, id_partida, valor, previsao)
  → listar_apostas_usuario(login, limite)


In [12]:
# Cole e rode essa consulta rápida para eu ver o estado real
from sqlalchemy.orm import Session

with Session(engine) as session:
    apostas = session.query(Aposta).all()
    print(f"Total de apostas no banco: {len(apostas)}")
    for a in apostas:
        print({
            "id_aposta":       a.id_aposta,
            "id_usuario":      a.id_usuario,
            "id_partida":      a.id_partida,
            "palpite":         a.palpite,
            "pontos_apostados": a.pontos_apostados,
            "status":          a.status,
        })

2026-07-14 21:55:59,917 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-14 21:55:59,927 INFO sqlalchemy.engine.Engine SELECT aposta.id_aposta AS aposta_id_aposta, aposta.id_usuario AS aposta_id_usuario, aposta.id_partida AS aposta_id_partida, aposta.palpite AS aposta_palpite, aposta.pontos_apostados AS aposta_pontos_apostados, aposta.multiplicador AS aposta_multiplicador, aposta.odd_aplicada AS aposta_odd_aplicada, aposta.retorno_potencial AS aposta_retorno_potencial, aposta.status AS aposta_status, aposta.data_hora AS aposta_data_hora 
FROM aposta


INFO:sqlalchemy.engine.Engine:SELECT aposta.id_aposta AS aposta_id_aposta, aposta.id_usuario AS aposta_id_usuario, aposta.id_partida AS aposta_id_partida, aposta.palpite AS aposta_palpite, aposta.pontos_apostados AS aposta_pontos_apostados, aposta.multiplicador AS aposta_multiplicador, aposta.odd_aplicada AS aposta_odd_aplicada, aposta.retorno_potencial AS aposta_retorno_potencial, aposta.status AS aposta_status, aposta.data_hora AS aposta_data_hora 
FROM aposta


2026-07-14 21:55:59,930 INFO sqlalchemy.engine.Engine [generated in 0.00368s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00368s] ()


Total de apostas no banco: 0
2026-07-14 21:55:59,933 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK


In [13]:
# Célula 8a — Testes completos das funções de aposta
from sqlalchemy.orm import Session
from sqlalchemy import select

print("=" * 60)
print("TESTES — FUNÇÕES DE APOSTA")
print("=" * 60)

LOGIN_TESTE  = "conta_teste_2026"
SALDO_MINIMO = 300.0

# ─────────────────────────────────────────────────────
# PREPARAÇÃO — garantir usuário ativo e saldo suficiente
# ─────────────────────────────────────────────────────

with Session(engine) as session:
    usuario = session.query(Usuario).filter_by(login=LOGIN_TESTE).first()
    if not usuario:
        print("❌ Usuário de teste não encontrado. Rode as células 6 e 7 primeiro.")
        raise SystemExit
    if usuario.status != "ativo":
        usuario.status = "ativo"
        session.commit()
        print(f"✅ Usuário '{LOGIN_TESTE}' reativado.")
    else:
        print(f"✅ Usuário '{LOGIN_TESTE}' ativo — saldo atual: {usuario.conta.saldo:.2f} pontos.")

saldo_info = consultar_saldo(LOGIN_TESTE)
if saldo_info["saldo"] < SALDO_MINIMO:
    recarga = round(SALDO_MINIMO - saldo_info["saldo"], 2)
    creditar_pontos(LOGIN_TESTE, recarga, descricao="Recarga para testes da Célula 8a")
    print(f"✅ Saldo recarregado em {recarga:.2f} pontos.")
else:
    print(f"✅ Saldo suficiente: {saldo_info['saldo']:.2f} pontos.")

# ─────────────────────────────────────────────────────
# Selecionar 2 partidas livres para o usuário
# ─────────────────────────────────────────────────────

with Session(engine) as session:
    usuario = session.query(Usuario).filter_by(login=LOGIN_TESTE).first()

    partidas_livres = (
        session.query(Partida)
        .filter(
            Partida.status.in_(["agendada", "aberta"]),
            ~Partida.id_partida.in_(
                select(Aposta.id_partida).where(Aposta.id_usuario == usuario.id_usuario)
            )
        )
        .order_by(Partida.num_jogo.asc())
        .limit(2)
        .all()
    )

    if len(partidas_livres) < 2:
        print("❌ Não há partidas suficientes para os testes.")
        raise SystemExit

    # Capturar dados das partidas ainda dentro da sessão
    p1_id       = partidas_livres[0].id_partida
    p1_num      = partidas_livres[0].num_jogo
    p1_odd_a    = float(partidas_livres[0].odd_a)
    p1_odd_emp  = float(partidas_livres[0].odd_empate)
    sel1a       = session.get(Selecao, partidas_livres[0].id_selecao_a)
    sel1b       = session.get(Selecao, partidas_livres[0].id_selecao_b)
    p1_nome_a   = sel1a.nome if sel1a else "Time A"
    p1_nome_b   = sel1b.nome if sel1b else "Time B"

    p2_id       = partidas_livres[1].id_partida
    p2_num      = partidas_livres[1].num_jogo
    p2_odd_b    = float(partidas_livres[1].odd_b)
    sel2a       = session.get(Selecao, partidas_livres[1].id_selecao_a)
    sel2b       = session.get(Selecao, partidas_livres[1].id_selecao_b)
    p2_nome_a   = sel2a.nome if sel2a else "Time A"
    p2_nome_b   = sel2b.nome if sel2b else "Time B"

print(f"\n✅ Partida 1 escolhida: Jogo {p1_num} — {p1_nome_a} x {p1_nome_b} (id={p1_id})")
print(f"✅ Partida 2 escolhida: Jogo {p2_num} — {p2_nome_a} x {p2_nome_b} (id={p2_id})")

# ─────────────────────────────────────────────────────
# TESTE 1 — Consultar partida para aposta
# ─────────────────────────────────────────────────────
print("\n" + "─" * 55)
print(f"📋 TESTE 1 — Consultar partida Jogo {p1_num}")
resultado = consultar_partida_para_aposta(p1_id)
assert resultado["sucesso"] is True,          f"FALHOU: {resultado}"
assert resultado["id_partida"] == p1_id,      "id_partida incorreto"
assert "odd_a" in resultado,                  "odd_a ausente"
assert "opcoes" in resultado,                 "opcoes ausente"
print(resultado)
print("✅ PASSOU")

# ─────────────────────────────────────────────────────
# TESTE 2 — Simular retorno: mandante vence
# ─────────────────────────────────────────────────────
print("\n" + "─" * 55)
print(f"📋 TESTE 2 — Simular retorno: {p1_nome_a} vence, 50 pontos")
resultado = simular_retorno_aposta(p1_id, 50, "A")
retorno_esperado = round(50 * p1_odd_a, 2)
assert resultado["sucesso"] is True,                         f"FALHOU: {resultado}"
assert resultado["palpite"] == "selecao_a",                  "palpite incorreto"
assert resultado["retorno_potencial"] == retorno_esperado,   f"retorno esperado {retorno_esperado}"
assert resultado["lucro_potencial"] == round(retorno_esperado - 50, 2), "lucro incorreto"
print(resultado)
print("✅ PASSOU")

# ─────────────────────────────────────────────────────
# TESTE 3 — Simular retorno: empate
# ─────────────────────────────────────────────────────
print("\n" + "─" * 55)
print(f"📋 TESTE 3 — Simular retorno: empate, 30 pontos")
resultado = simular_retorno_aposta(p1_id, 30, "EMPATE")
retorno_emp = round(30 * p1_odd_emp, 2)
assert resultado["sucesso"] is True,                       f"FALHOU: {resultado}"
assert resultado["palpite"] == "empate",                   "palpite incorreto"
assert resultado["retorno_potencial"] == retorno_emp,      f"retorno esperado {retorno_emp}"
print(resultado)
print("✅ PASSOU")

# ─────────────────────────────────────────────────────
# TESTE 4 — Criar aposta real 1 (mandante vence)
# ─────────────────────────────────────────────────────
print("\n" + "─" * 55)
print(f"📋 TESTE 4 — Criar aposta: {p1_nome_a} vence, 50 pontos")
saldo_antes_t4 = consultar_saldo(LOGIN_TESTE)["saldo"]
resultado = criar_aposta(LOGIN_TESTE, p1_id, 50, "A")
saldo_depois_t4 = consultar_saldo(LOGIN_TESTE)["saldo"]
assert resultado["sucesso"] is True,                          f"FALHOU: {resultado}"
assert resultado["palpite"] == "selecao_a",                   "palpite incorreto"
assert resultado["status_aposta"] == "ativa",                 "status incorreto"
assert resultado["multiplicador"] == 1,                       "multiplicador incorreto"
assert resultado["saldo_atual"] == round(saldo_antes_t4 - 50, 2), "saldo incorreto"
assert saldo_depois_t4 == resultado["saldo_atual"],           "saldo no banco diverge"
print(resultado)
print("✅ PASSOU")

# ─────────────────────────────────────────────────────
# TESTE 5 — Criar aposta real 2 (visitante vence)
# ─────────────────────────────────────────────────────
print("\n" + "─" * 55)
print(f"📋 TESTE 5 — Criar aposta: {p2_nome_b} vence, 30 pontos")
saldo_antes_t5 = consultar_saldo(LOGIN_TESTE)["saldo"]
resultado = criar_aposta(LOGIN_TESTE, p2_id, 30, "B")
saldo_depois_t5 = consultar_saldo(LOGIN_TESTE)["saldo"]
assert resultado["sucesso"] is True,                          f"FALHOU: {resultado}"
assert resultado["palpite"] == "selecao_b",                   "palpite incorreto"
assert resultado["status_aposta"] == "ativa",                 "status incorreto"
assert resultado["saldo_atual"] == round(saldo_antes_t5 - 30, 2), "saldo incorreto"
assert saldo_depois_t5 == resultado["saldo_atual"],           "saldo no banco diverge"
print(resultado)
print("✅ PASSOU")

# ─────────────────────────────────────────────────────
# TESTE 6 — Aposta duplicada (mesma partida)
# ─────────────────────────────────────────────────────
print("\n" + "─" * 55)
print(f"📋 TESTE 6 — Aposta duplicada na Partida {p1_id}")
resultado = criar_aposta(LOGIN_TESTE, p1_id, 20, "B")
assert resultado["sucesso"] is False,     f"Deveria ter falhado: {resultado}"
assert "aposta registrada" in resultado["erro"].lower(), f"Mensagem inesperada: {resultado['erro']}"
print(resultado)
print("✅ PASSOU — duplicidade bloqueada corretamente")

# ─────────────────────────────────────────────────────
# TESTE 7 — Previsão inválida
# ─────────────────────────────────────────────────────
print("\n" + "─" * 55)
print("📋 TESTE 7 — Previsão inválida ('XYZ')")
resultado = criar_aposta(LOGIN_TESTE, p1_id, 10, "XYZ")
assert resultado["sucesso"] is False,    f"Deveria ter falhado: {resultado}"
assert "inválida" in resultado["erro"],  f"Mensagem inesperada: {resultado['erro']}"
print(resultado)
print("✅ PASSOU — previsão inválida bloqueada")

# ─────────────────────────────────────────────────────
# TESTE 8 — Partida inexistente
# ─────────────────────────────────────────────────────
print("\n" + "─" * 55)
print("📋 TESTE 8 — Partida inexistente (id=999999)")
resultado = criar_aposta(LOGIN_TESTE, 999999, 10, "A")
assert resultado["sucesso"] is False,       f"Deveria ter falhado: {resultado}"
assert "não encontrada" in resultado["erro"].lower(), f"Mensagem inesperada: {resultado['erro']}"
print(resultado)
print("✅ PASSOU — partida inexistente bloqueada")

# ─────────────────────────────────────────────────────
# TESTE 9 — Saldo insuficiente
# ─────────────────────────────────────────────────────
print("\n" + "─" * 55)
print("📋 TESTE 9 — Saldo insuficiente (aposta de 99999 pontos)")

# Buscar uma partida ainda livre para esse teste
with Session(engine) as session:
    usuario = session.query(Usuario).filter_by(login=LOGIN_TESTE).first()
    p_livre = (
        session.query(Partida)
        .filter(
            Partida.status.in_(["agendada", "aberta"]),
            ~Partida.id_partida.in_(
                select(Aposta.id_partida).where(Aposta.id_usuario == usuario.id_usuario)
            )
        )
        .order_by(Partida.num_jogo.asc())
        .first()
    )
    p_livre_id = p_livre.id_partida if p_livre else None

if p_livre_id:
    resultado = criar_aposta(LOGIN_TESTE, p_livre_id, 99999, "A")
    assert resultado["sucesso"] is False,         f"Deveria ter falhado: {resultado}"
    assert "insuficiente" in resultado["erro"].lower(), f"Mensagem inesperada: {resultado['erro']}"
    print(resultado)
    print("✅ PASSOU — saldo insuficiente bloqueado")
else:
    print("⚠️  Nenhuma partida livre para esse teste.")

# ─────────────────────────────────────────────────────
# TESTE 10 — Listar apostas do usuário
# ─────────────────────────────────────────────────────
print("\n" + "─" * 55)
print("📋 TESTE 10 — Listar apostas")
resultado = listar_apostas_usuario(LOGIN_TESTE, limite=10)
assert resultado["sucesso"] is True,          f"FALHOU: {resultado}"
assert resultado["total_apostas"] == 2,       f"Esperado 2 apostas, obtido: {resultado['total_apostas']}"
for aposta in resultado["apostas"]:
    assert aposta["palpite"] in ("selecao_a", "selecao_b", "empate"), \
        f"Palpite inválido no histórico: {aposta['palpite']}"
    assert aposta["status"] in ("ativa", "ganha", "perdida", "reembolsada", "anulada"), \
        f"Status inválido no histórico: {aposta['status']}"
print(resultado)
print("✅ PASSOU")

# ─────────────────────────────────────────────────────
# TESTE 11 — Saldo final consistente
# ─────────────────────────────────────────────────────
print("\n" + "─" * 55)
print("📋 TESTE 11 — Consistência do saldo final")
saldo_final = consultar_saldo(LOGIN_TESTE)["saldo"]
saldo_esperado = round(SALDO_MINIMO - 50 - 30, 2)  # 300 - 50 - 30 = 220
assert saldo_final == saldo_esperado, \
    f"Saldo esperado: {saldo_esperado}, obtido: {saldo_final}"
print(f"Saldo final: {saldo_final:.2f} pontos (esperado: {saldo_esperado:.2f})")
print("✅ PASSOU")

# ─────────────────────────────────────────────────────
# TESTE 12 — Movimentações registradas
# ─────────────────────────────────────────────────────
print("\n" + "─" * 55)
print("📋 TESTE 12 — Movimentações registradas")
resultado = listar_movimentacoes(LOGIN_TESTE, limite=5)
assert resultado["sucesso"] is True,                  f"FALHOU: {resultado}"
assert resultado["total_movimentacoes"] >= 2,         "Menos movimentações do que o esperado"
debitos = [m for m in resultado["historico"] if m["tipo_movimentacao"] == "debito"]
assert len(debitos) >= 2, f"Esperado pelo menos 2 débitos, obtido: {len(debitos)}"
print(f"Total de movimentações recentes: {resultado['total_movimentacoes']}")
for m in resultado["historico"]:
    print(f"  [{m['tipo_movimentacao'].upper()}] {m['pontos']} pts — {m['descricao']}")
print("✅ PASSOU")

# ─────────────────────────────────────────────────────
# RESUMO FINAL
# ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("RESUMO DOS TESTES")
print("=" * 60)
print(f"  Apostas criadas:   2")
print(f"  Saldo final:       {consultar_saldo(LOGIN_TESTE)['saldo']:.2f} pontos")
print(f"  Duplicidade:       bloqueada ✅")
print(f"  Previsão inválida: bloqueada ✅")
print(f"  Partida inválida:  bloqueada ✅")
print(f"  Saldo insuficiente:bloqueada ✅")
print("=" * 60)
print("TESTES CONCLUÍDOS")
print("=" * 60)

TESTES — FUNÇÕES DE APOSTA
2026-07-14 21:55:59,977 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-14 21:55:59,979 INFO sqlalchemy.engine.Engine SELECT usuario.id_usuario AS usuario_id_usuario, usuario.nome AS usuario_nome, usuario.email AS usuario_email, usuario.cpf AS usuario_cpf, usuario.data_nascimento AS usuario_data_nascimento, usuario.login AS usuario_login, usuario.senha_hash AS usuario_senha_hash, usuario.status AS usuario_status, usuario.tipo_usuario AS usuario_tipo_usuario 
FROM usuario 
WHERE usuario.login = ?
 LIMIT ? OFFSET ?


INFO:sqlalchemy.engine.Engine:SELECT usuario.id_usuario AS usuario_id_usuario, usuario.nome AS usuario_nome, usuario.email AS usuario_email, usuario.cpf AS usuario_cpf, usuario.data_nascimento AS usuario_data_nascimento, usuario.login AS usuario_login, usuario.senha_hash AS usuario_senha_hash, usuario.status AS usuario_status, usuario.tipo_usuario AS usuario_tipo_usuario 
FROM usuario 
WHERE usuario.login = ?
 LIMIT ? OFFSET ?


2026-07-14 21:55:59,981 INFO sqlalchemy.engine.Engine [cached since 0.9423s ago] ('conta_teste_2026', 1, 0)


...

2026-07-14 21:56:00,949 INFO sqlalchemy.engine.Engine [cached since 1.391s ago] (2, 1, 0)


INFO:sqlalchemy.engine.Engine:[cached since 1.391s ago] (2, 1, 0)


2026-07-14 21:56:00,951 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK


  Saldo final:       220.00 pontos
  Duplicidade:       bloqueada ✅
  Previsão inválida: bloqueada ✅
  Partida inválida:  bloqueada ✅
  Saldo insuficiente:bloqueada ✅
TESTES CONCLUÍDOS


In [14]:
# Célula 9 — Liquidação de Apostas
# Funções: liquidar_partida | cancelar_partida | consultar_resultado_partida
#
# Regras:
#   palpite correto  → status 'ganha'      + credito do retorno_potencial
#   palpite errado   → status 'perdida'    (sem movimentação)
#   partida cancelada→ status 'reembolsada'+ estorno dos pontos_apostados
#   reliquidação     → bloqueada (partida já 'encerrada' ou 'cancelada')

from sqlalchemy.orm import Session
from datetime import datetime, timezone


# ─────────────────────────────────────────────────────────────
# HELPER INTERNO
# ─────────────────────────────────────────────────────────────

def _resultado_real(gols_a: int, gols_b: int) -> str:
    """Converte placar em resultado normalizado."""
    if gols_a > gols_b:
        return "selecao_a"
    elif gols_b > gols_a:
        return "selecao_b"
    else:
        return "empate"


def _nomes_partida(session, partida) -> tuple:
    """Retorna (nome_a, nome_b) de uma partida."""
    sel_a = session.get(Selecao, partida.id_selecao_a)
    sel_b = session.get(Selecao, partida.id_selecao_b)
    return (
        sel_a.nome if sel_a else "?",
        sel_b.nome if sel_b else "?",
    )


# ─────────────────────────────────────────────────────────────
# FUNÇÃO 1 — liquidar_partida
# ─────────────────────────────────────────────────────────────

def liquidar_partida(id_partida: int, gols_a: int, gols_b: int) -> dict:
    """
    Encerra uma partida com o placar real e liquida todas as apostas ativas.

    Parâmetros:
        id_partida : id da partida a encerrar
        gols_a     : gols da selecao_a (mandante)
        gols_b     : gols da selecao_b (visitante)

    Retorno:
        dict com sucesso, placar, resultado_real e detalhes por aposta
    """
    # Validações básicas
    if not isinstance(gols_a, int) or not isinstance(gols_b, int):
        return {"sucesso": False, "erro": "gols_a e gols_b devem ser inteiros."}
    if gols_a < 0 or gols_b < 0:
        return {"sucesso": False, "erro": "Gols não podem ser negativos."}

    with Session(engine) as session:
        try:
            partida = session.get(Partida, id_partida)
            if not partida:
                return {"sucesso": False, "erro": f"Partida {id_partida} não encontrada."}
            if partida.status == "encerrada":
                return {"sucesso": False, "erro": f"Partida {id_partida} já foi encerrada anteriormente."}
            if partida.status == "cancelada":
                return {"sucesso": False, "erro": f"Partida {id_partida} está cancelada. Não pode ser liquidada."}

            nome_a, nome_b = _nomes_partida(session, partida)
            resultado_real = _resultado_real(gols_a, gols_b)

            # Atualizar partida
            partida.status = "encerrada"
            partida.gols_a = gols_a
            partida.gols_b = gols_b
            session.flush()

            # Buscar apostas ativas desta partida
            apostas_ativas = (
                session.query(Aposta)
                .filter_by(id_partida=id_partida, status="ativa")
                .all()
            )

            ganhas   = []
            perdidas = []
            agora    = datetime.now(timezone.utc)

            for aposta in apostas_ativas:
                conta   = session.query(Conta_Pontos).filter_by(id_usuario=aposta.id_usuario).first()
                usuario = session.get(Usuario, aposta.id_usuario)
                login   = usuario.login if usuario else "?"

                if aposta.palpite == resultado_real:
                    # ── GANHA ──────────────────────────────────
                    aposta.status   = "ganha"
                    saldo_anterior  = conta.saldo
                    conta.saldo     = round(conta.saldo + aposta.retorno_potencial, 2)
                    conta.data_atualizacao = agora

                    session.add(Movimentacao_Pontos(
                        id_conta          = conta.id_conta,
                        id_aposta         = aposta.id_aposta,
                        tipo_movimentacao = "credito",
                        pontos            = aposta.retorno_potencial,
                        saldo_anterior    = saldo_anterior,
                        saldo_atual       = conta.saldo,
                        data_hora         = agora,
                        descricao         = (
                            f"Aposta ganha — Jogo {partida.num_jogo}: "
                            f"{nome_a} {gols_a}x{gols_b} {nome_b} | "
                            f"Retorno: {aposta.retorno_potencial:.2f} pts"
                        ),
                    ))

                    ganhas.append({
                        "id_aposta":         aposta.id_aposta,
                        "login":             login,
                        "palpite":           aposta.palpite,
                        "retorno_creditado": aposta.retorno_potencial,
                        "saldo_apos":        conta.saldo,
                    })

                else:
                    # ── PERDIDA ─────────────────────────────────
                    aposta.status = "perdida"

                    perdidas.append({
                        "id_aposta":      aposta.id_aposta,
                        "login":          login,
                        "palpite":        aposta.palpite,
                        "pontos_perdidos": aposta.pontos_apostados,
                    })

            session.commit()

            return {
                "sucesso":                    True,
                "id_partida":                 id_partida,
                "num_jogo":                   partida.num_jogo,
                "partida":                    f"{nome_a} x {nome_b}",
                "placar":                     f"{gols_a}x{gols_b}",
                "resultado_real":             resultado_real,
                "total_apostas_liquidadas":   len(apostas_ativas),
                "ganhas":                     len(ganhas),
                "perdidas":                   len(perdidas),
                "detalhes_ganhas":            ganhas,
                "detalhes_perdidas":          perdidas,
            }

        except Exception as e:
            session.rollback()
            return {"sucesso": False, "erro": f"Erro na liquidação: {str(e)}"}


# ─────────────────────────────────────────────────────────────
# FUNÇÃO 2 — cancelar_partida
# ─────────────────────────────────────────────────────────────

def cancelar_partida(id_partida: int, motivo: str = "Cancelamento administrativo") -> dict:
    """
    Cancela uma partida e reembolsa (estorno) todos os apostadores com apostas ativas.
    """
    with Session(engine) as session:
        try:
            partida = session.get(Partida, id_partida)
            if not partida:
                return {"sucesso": False, "erro": f"Partida {id_partida} não encontrada."}
            if partida.status == "encerrada":
                return {"sucesso": False, "erro": "Partida já encerrada. Não pode ser cancelada."}
            if partida.status == "cancelada":
                return {"sucesso": False, "erro": "Partida já está cancelada."}

            nome_a, nome_b = _nomes_partida(session, partida)

            partida.status = "cancelada"
            session.flush()

            apostas_ativas = (
                session.query(Aposta)
                .filter_by(id_partida=id_partida, status="ativa")
                .all()
            )

            reembolsos = []
            agora      = datetime.now(timezone.utc)

            for aposta in apostas_ativas:
                conta   = session.query(Conta_Pontos).filter_by(id_usuario=aposta.id_usuario).first()
                usuario = session.get(Usuario, aposta.id_usuario)
                login   = usuario.login if usuario else "?"

                aposta.status          = "reembolsada"
                saldo_anterior         = conta.saldo
                conta.saldo            = round(conta.saldo + aposta.pontos_apostados, 2)
                conta.data_atualizacao = agora

                session.add(Movimentacao_Pontos(
                    id_conta          = conta.id_conta,
                    id_aposta         = aposta.id_aposta,
                    tipo_movimentacao = "estorno",
                    pontos            = aposta.pontos_apostados,
                    saldo_anterior    = saldo_anterior,
                    saldo_atual       = conta.saldo,
                    data_hora         = agora,
                    descricao         = (
                        f"Reembolso — Jogo {partida.num_jogo}: "
                        f"{nome_a} x {nome_b} cancelado | {motivo}"
                    ),
                ))

                reembolsos.append({
                    "id_aposta":          aposta.id_aposta,
                    "login":              login,
                    "pontos_reembolsados": aposta.pontos_apostados,
                    "saldo_apos":         conta.saldo,
                })

            session.commit()

            return {
                "sucesso":         True,
                "id_partida":      id_partida,
                "num_jogo":        partida.num_jogo,
                "partida":         f"{nome_a} x {nome_b}",
                "status":          "cancelada",
                "motivo":          motivo,
                "total_reembolsos": len(reembolsos),
                "reembolsos":      reembolsos,
            }

        except Exception as e:
            session.rollback()
            return {"sucesso": False, "erro": f"Erro no cancelamento: {str(e)}"}


# ─────────────────────────────────────────────────────────────
# FUNÇÃO 3 — consultar_resultado_partida
# ─────────────────────────────────────────────────────────────

def consultar_resultado_partida(id_partida: int) -> dict:
    """
    Retorna o estado atual de uma partida e o resumo das apostas por status.
    """
    with Session(engine) as session:
        partida = session.get(Partida, id_partida)
        if not partida:
            return {"sucesso": False, "erro": f"Partida {id_partida} não encontrada."}

        nome_a, nome_b = _nomes_partida(session, partida)

        apostas = session.query(Aposta).filter_by(id_partida=id_partida).all()

        resumo = {}
        for a in apostas:
            resumo[a.status] = resumo.get(a.status, 0) + 1

        return {
            "sucesso":          True,
            "id_partida":       id_partida,
            "num_jogo":         partida.num_jogo,
            "partida":          f"{nome_a} x {nome_b}",
            "status_partida":   partida.status,
            "placar":           f"{partida.gols_a}x{partida.gols_b}" if partida.status == "encerrada" else "—",
            "resultado_real":   _resultado_real(partida.gols_a, partida.gols_b) if partida.status == "encerrada" else "—",
            "total_apostas":    len(apostas),
            "apostas_por_status": resumo,
        }


print("✅ Célula 9 carregada — funções de liquidação disponíveis:")
print("   → liquidar_partida(id_partida, gols_a, gols_b)")
print("   → cancelar_partida(id_partida, motivo)")
print("   → consultar_resultado_partida(id_partida)")

✅ Célula 9 carregada — funções de liquidação disponíveis:
   → liquidar_partida(id_partida, gols_a, gols_b)
   → cancelar_partida(id_partida, motivo)
   → consultar_resultado_partida(id_partida)


In [15]:
def criar_aposta(session, id_usuario: int, id_partida: int, palpite: str, pontos: float, multiplicador: int = 1):
    """
    Cria uma aposta validando saldo e registrando movimentação de débito.
    Retorna dict: {sucesso: bool, id_aposta: int, erro: str}
    Observações:
      - Verifica saldo antes do débito (previne ck_conta_saldo).
      - Usa session.flush() para obter id_aposta antes de inserir movimentacao.
      - Não comita; deixa quem chamou fazer commit/rollback.
    """
    # Validações
    if palpite not in ("selecao_a","empate","selecao_b"):
        return {"sucesso": False, "erro": "Palpite inválido."}
    if pontos <= 0:
        return {"sucesso": False, "erro": "Pontos devem ser > 0."}
    if multiplicador not in (1,2,3,4,5):
        return {"sucesso": False, "erro": "Multiplicador inválido."}

    # Buscar conta do usuário
    conta = session.query(Conta_Pontos).filter_by(id_usuario=id_usuario).with_for_update().first()
    if not conta:
        return {"sucesso": False, "erro": "Conta do usuário não encontrada."}

    # Verificar saldo suficiente
    if round(conta.saldo,2) < round(pontos,2):
        return {"sucesso": False, "erro": f"Saldo insuficiente. Saldo atual: {conta.saldo:.2f}."}

    # Buscar partida e odds
    partida = session.get(Partida, id_partida)
    if not partida:
        return {"sucesso": False, "erro": "Partida não encontrada."}
    if partida.status not in ("agendada","ao_vivo"):
        return {"sucesso": False, "erro": f"Partida com status '{partida.status}' não aceita apostas."}

    # Determinar odd aplicada
    if palpite == "selecao_a":
        odd = partida.odd_a
    elif palpite == "selecao_b":
        odd = partida.odd_b
    else:
        odd = partida.odd_empate

    retorno = round(pontos * odd * multiplicador, 2)

    # Criar aposta (sem commit ainda)
    aposta = Aposta(
        id_usuario = id_usuario,
        id_partida = id_partida,
        palpite = palpite,
        pontos_apostados = pontos,
        multiplicador = multiplicador,
        odd_aplicada = odd,
        retorno_potencial = retorno,
        status = "ativa"
    )
    session.add(aposta)
    session.flush()  # garante aposta.id_aposta

    # Debitar conta e criar movimentacao
    saldo_ant = conta.saldo
    conta.saldo = round(conta.saldo - pontos, 2)
    conta.data_atualizacao = datetime.now(timezone.utc)

    mov = Movimentacao_Pontos(
        id_conta = conta.id_conta,
        id_aposta = aposta.id_aposta,
        tipo_movimentacao = "debito",
        pontos = pontos,
        saldo_anterior = saldo_ant,
        saldo_atual = conta.saldo,
        data_hora = datetime.now(timezone.utc),
        descricao = f"Aposta Jogo {partida.num_jogo}: { _nomes_partida(session, partida)[0] } | valor {pontos}"
    )
    session.add(mov)

    return {"sucesso": True, "id_aposta": aposta.id_aposta}

In [16]:
from sqlalchemy.orm import Session

login_teste = "conta_teste_2026"

with Session(engine) as s:
    usuario = s.query(Usuario).filter_by(login=login_teste).first()
    if not usuario:
        raise RuntimeError(f"Usuário de teste '{login_teste}' não encontrado.")
    conta = s.query(Conta_Pontos).filter_by(id_usuario=usuario.id_usuario).first()
    print(f"Usuário: {usuario.login} (id={usuario.id_usuario}) — saldo: {conta.saldo:.2f} pts")
usuario = s.query(Usuario).filter_by(login=login_teste).first()

if not usuario:
    raise RuntimeError(f"Usuário de teste '{login_teste}' não encontrado.")

conta = s.query(Conta_Pontos).filter_by(id_usuario=usuario.id_usuario).first()
if not conta:
    raise RuntimeError(f"Conta do usuário {usuario.id_usuario} não encontrada.")

saldo_anterior = round(conta.saldo, 2)
recarga = 100.0
conta.saldo = round(conta.saldo + recarga, 2)
conta.data_atualizacao = datetime.now(timezone.utc)

mov = Movimentacao_Pontos(
    id_conta = conta.id_conta,
    id_aposta = None,
    tipo_movimentacao = "credito",
    pontos = recarga,
    saldo_anterior = saldo_anterior,
    saldo_atual = conta.saldo,
    data_hora = datetime.now(timezone.utc),
    descricao = "Recarga para testes — Célula 9a"
)
s.add(mov)
s.commit()

print(f"Recarga efetuada: +{recarga:.2f} pts")
print(f"Saldo anterior: {saldo_anterior:.2f} pts → Saldo atual: {conta.saldo:.2f} pts")
print(f"Movimentação criada: id_movimentacao={mov.id_movimentacao}")


2026-07-14 21:56:01,113 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-14 21:56:01,115 INFO sqlalchemy.engine.Engine SELECT usuario.id_usuario AS usuario_id_usuario, usuario.nome AS usuario_nome, usuario.email AS usuario_email, usuario.cpf AS usuario_cpf, usuario.data_nascimento AS usuario_data_nascimento, usuario.login AS usuario_login, usuario.senha_hash AS usuario_senha_hash, usuario.status AS usuario_status, usuario.tipo_usuario AS usuario_tipo_usuario 
FROM usuario 
WHERE usuario.login = ?
 LIMIT ? OFFSET ?


INFO:sqlalchemy.engine.Engine:SELECT usuario.id_usuario AS usuario_id_usuario, usuario.nome AS usuario_nome, usuario.email AS usuario_email, usuario.cpf AS usuario_cpf, usuario.data_nascimento AS usuario_data_nascimento, usuario.login AS usuario_login, usuario.senha_hash AS usuario_senha_hash, usuario.status AS usuario_status, usuario.tipo_usuario AS usuario_tipo_usuario 
FROM usuario 
WHERE usuario.login = ?
 LIMIT ? OFFSET ?


2026-07-14 21:56:01,120 INFO sqlalchemy.engine.Engine [cached since 2.081s ago] ('conta_teste_2026', 1, 0)


...

Saldo anterior: 220.00 pts → Saldo atual: 320.00 pts
2026-07-14 21:56:01,226 INFO sqlalchemy.engine.Engine SELECT movimentacao_pontos.id_movimentacao AS movimentacao_pontos_id_movimentacao, movimentacao_pontos.id_conta AS movimentacao_pontos_id_conta, movimentacao_pontos.id_aposta AS movimentacao_pontos_id_aposta, movimentacao_pontos.tipo_movimentacao AS movimentacao_pontos_tipo_movimentacao, movimentacao_pontos.pontos AS movimentacao_pontos_pontos, movimentacao_pontos.saldo_anterior AS movimentacao_pontos_saldo_anterior, movimentacao_pontos.saldo_atual AS movimentacao_pontos_saldo_atual, movimentacao_pontos.data_hora AS movimentacao_pontos_data_hora, movimentacao_pontos.descricao AS movimentacao_pontos_descricao 
FROM movimentacao_pontos 
WHERE movimentacao_pontos.id_movimentacao = ?


INFO:sqlalchemy.engine.Engine:SELECT movimentacao_pontos.id_movimentacao AS movimentacao_pontos_id_movimentacao, movimentacao_pontos.id_conta AS movimentacao_pontos_id_conta, movimentacao_pontos.id_aposta AS movimentacao_pontos_id_aposta, movimentacao_pontos.tipo_movimentacao AS movimentacao_pontos_tipo_movimentacao, movimentacao_pontos.pontos AS movimentacao_pontos_pontos, movimentacao_pontos.saldo_anterior AS movimentacao_pontos_saldo_anterior, movimentacao_pontos.saldo_atual AS movimentacao_pontos_saldo_atual, movimentacao_pontos.data_hora AS movimentacao_pontos_data_hora, movimentacao_pontos.descricao AS movimentacao_pontos_descricao 
FROM movimentacao_pontos 
WHERE movimentacao_pontos.id_movimentacao = ?


2026-07-14 21:56:01,233 INFO sqlalchemy.engine.Engine [generated in 0.00673s] (9,)


INFO:sqlalchemy.engine.Engine:[generated in 0.00673s] (9,)


Movimentação criada: id_movimentacao=9


In [17]:
# Célula 9a — Testes auto-suficientes de liquidação (cole inteiro e rode)
from sqlalchemy.orm import Session
from datetime import datetime, timezone

# Helper seguro para criar aposta (verifica saldo, debita, registra movimentação)
def criar_aposta(session, id_usuario: int, id_partida: int, palpite: str, pontos: float, multiplicador: int = 1):
    """
    Cria aposta atomically dentro da sessão fornecida.
    Retorna dict: {sucesso: bool, id_aposta: int, erro: str}
    NOTA: não faz commit; quem chamar decide commit/rollback.
    """
    # Validações
    if palpite not in ("selecao_a", "empate", "selecao_b"):
        return {"sucesso": False, "erro": "Palpite inválido"}
    if pontos <= 0:
        return {"sucesso": False, "erro": "Pontos inválidos"}

    # Garantir que partida e conta existam (usa a mesma sessão)
    partida = session.get(Partida, id_partida)
    if not partida:
        return {"sucesso": False, "erro": f"Partida {id_partida} não encontrada"}
    conta = session.query(Conta_Pontos).filter_by(id_usuario=id_usuario).first()
    if not conta:
        return {"sucesso": False, "erro": f"Conta do usuário {id_usuario} não encontrada"}

    # Verificar saldo
    if round(conta.saldo, 2) < round(pontos, 2):
        return {"sucesso": False, "erro": f"Saldo insuficiente. Saldo atual: {conta.saldo:.2f}"}

    # Calcular odd aplicada e retorno potencial a partir da partida
    if palpite == "selecao_a":
        odd = partida.odd_a
    elif palpite == "selecao_b":
        odd = partida.odd_b
    else:
        odd = partida.odd_empate

    retorno = round(pontos * odd * multiplicador, 2)

    # Criar aposta e debitar saldo — dentro da mesma sessão
    agora = datetime.now(timezone.utc)
    aposta = Aposta(
        id_usuario = id_usuario,
        id_partida = id_partida,
        palpite = palpite,
        pontos_apostados = pontos,
        multiplicador = multiplicador,
        odd_aplicada = odd,
        retorno_potencial = retorno,
        status = "ativa",
        data_hora = agora,
    )
    session.add(aposta)
    session.flush()  # assegura que aposta.id_aposta exista

    # Debitar conta e registrar movimentação
    saldo_anterior = conta.saldo
    conta.saldo = round(conta.saldo - pontos, 2)
    conta.data_atualizacao = agora

    mov = Movimentacao_Pontos(
        id_conta = conta.id_conta,
        id_aposta = aposta.id_aposta,
        tipo_movimentacao = "debito",
        pontos = pontos,
        saldo_anterior = saldo_anterior,
        saldo_atual = conta.saldo,
        data_hora = agora,
        descricao = f"Aposta Jogo {partida.num_jogo}: {session.get(Selecao, partida.id_selecao_a).nome} x {session.get(Selecao, partida.id_selecao_b).nome} | valor {pontos}",
    )
    session.add(mov)

    return {"sucesso": True, "id_aposta": aposta.id_aposta, "retorno_potencial": retorno}

# Helper para listar últimas movimentações (por login)
def listar_movimentacoes(login: str, limite: int = 12):
    with Session(engine) as session:
        usuario = session.query(Usuario).filter_by(login=login).first()
        if not usuario:
            return {"sucesso": False, "erro": "usuario_nao_encontrado"}
        conta = session.query(Conta_Pontos).filter_by(id_usuario=usuario.id_usuario).first()
        if not conta:
            return {"sucesso": False, "erro": "conta_nao_encontrada"}
        movs = (
            session.query(Movimentacao_Pontos)
            .filter_by(id_conta=conta.id_conta)
            .order_by(Movimentacao_Pontos.data_hora.desc())
            .limit(limite)
            .all()
        )
        historico = []
        for m in movs:
            historico.append({
                "tipo_movimentacao": m.tipo_movimentacao,
                "pontos": m.pontos,
                "saldo_anterior": m.saldo_anterior,
                "saldo_atual": m.saldo_atual,
                "descricao": (m.descricao or "")[:200],
                "data_hora": m.data_hora.isoformat(),
                "id_aposta": m.id_aposta,
            })
        return {"sucesso": True, "historico": historico}

# Início dos testes auto-suficientes
print("=" * 70)
print("TESTES — LIQUIDAÇÃO DE APOSTAS (AUTO-SUFICIENTE)")
print("=" * 70)

LOGIN_TESTE = "conta_teste_2026"

with Session(engine) as session:
    usuario = session.query(Usuario).filter_by(login=LOGIN_TESTE).first()
    assert usuario is not None, "Usuário de teste não encontrado"
    id_usuario = usuario.id_usuario

    conta = session.query(Conta_Pontos).filter_by(id_usuario=id_usuario).first()
    assert conta is not None, "Conta do usuário de teste não encontrada"
    saldo_inicial = round(conta.saldo, 2)
    print(f"Saldo inicial: {saldo_inicial:.2f} pts")

    # Selecionar 4 partidas agendadas livres (sem apostas do usuário)
    partidas_livres = (
        session.query(Partida)
        .filter(
            Partida.status == "agendada",
            ~Partida.id_partida.in_(
                session.query(Aposta.id_partida).filter(Aposta.id_usuario == id_usuario)
            )
        )
        .order_by(Partida.num_jogo.asc())
        .limit(4)
        .all()
    )
    assert len(partidas_livres) >= 4, "Não há partidas livres suficientes para os testes"

    # Mapear ids (usaremos apenas ids — nunca passamos instâncias entre sessões)
    id_p1 = partidas_livres[0].id_partida
    id_p2 = partidas_livres[1].id_partida
    id_p3 = partidas_livres[2].id_partida
    id_p4 = partidas_livres[3].id_partida

    print("Partidas selecionadas (ids):", [id_p1, id_p2, id_p3, id_p4])

    # Valores de aposta conservadores (para evitar falta de saldo)
    v1 = 1.0   # p1 será ganha
    v2 = 2.0   # p2 será perdida
    v3 = 1.5   # p3 será cancelada (reembolso)
    v4 = 1.0   # p4 será empate (ganha empate)

    # Criar 4 apostas EM UMA ÚNICA TRANSAÇÃO (para garantir atomicidade)
    apostas_ids = []
    r = criar_aposta(session, id_usuario, id_p1, "selecao_a", v1, multiplicador=1)
    assert r["sucesso"], f"Falha criando aposta p1: {r}"
    apostas_ids.append(r["id_aposta"])
    id_ap1 = r["id_aposta"]

    r = criar_aposta(session, id_usuario, id_p2, "selecao_b", v2, multiplicador=1)
    assert r["sucesso"], f"Falha criando aposta p2: {r}"
    apostas_ids.append(r["id_aposta"])
    id_ap2 = r["id_aposta"]

    r = criar_aposta(session, id_usuario, id_p3, "empate", v3, multiplicador=1)
    assert r["sucesso"], f"Falha criando aposta p3: {r}"
    apostas_ids.append(r["id_aposta"])
    id_ap3 = r["id_aposta"]

    r = criar_aposta(session, id_usuario, id_p4, "empate", v4, multiplicador=1)
    assert r["sucesso"], f"Falha criando aposta p4: {r}"
    apostas_ids.append(r["id_aposta"])
    id_ap4 = r["id_aposta"]

    # Commit das apostas e débitos
    session.commit()
    print("Apostas criadas com sucesso:", apostas_ids)

# 1) Liquidar p1 como vitória da seleção A (ganha)
r1 = liquidar_partida(id_partida=id_p1, gols_a=2, gols_b=0)
assert r1["sucesso"], f"Falha liquidação p1: {r1}"
print("Liquidação p1:", r1["resultado_real"], "ganhas:", r1["ganhas"], "perdidas:", r1["perdidas"])

# 2) Liquidar p2 como vitória da seleção A (aposta p2 errada -> perdida)
r2 = liquidar_partida(id_partida=id_p2, gols_a=1, gols_b=0)
assert r2["sucesso"], f"Falha liquidação p2: {r2}"
print("Liquidação p2:", r2["resultado_real"], "ganhas:", r2["ganhas"], "perdidas:", r2["perdidas"])

# 3) Cancelar p3 (reembolso)
r3 = cancelar_partida(id_partida=id_p3, motivo="Teste de cancelamento")
assert r3["sucesso"], f"Falha cancelamento p3: {r3}"
print("Cancelamento p3:", r3["total_reembolsos"], "reembolsos registrados")

# 4) Liquidar p4 como empate (ganha)
r4 = liquidar_partida(id_partida=id_p4, gols_a=1, gols_b=1)
assert r4["sucesso"], f"Falha liquidação p4: {r4}"
print("Liquidação p4:", r4["resultado_real"], "ganhas:", r4["ganhas"], "perdidas:", r4["perdidas"])

# Verificações finais de saldo e movimentações
with Session(engine) as session:
    conta = session.query(Conta_Pontos).filter_by(id_usuario=id_usuario).first()
    saldo_final = round(conta.saldo, 2)
    print("\nSaldo final (conta):", saldo_final)

    movs = (
        session.query(Movimentacao_Pontos)
        .filter_by(id_conta=conta.id_conta)
        .order_by(Movimentacao_Pontos.data_hora.desc())
        .limit(20)
        .all()
    )
    print("\nÚltimas movimentações:")
    for m in movs[:12]:
        print(f"  [{m.tipo_movimentacao.upper():7}] {m.pontos:6.1f} pts — {m.descricao[:120]}")

print("\n" + "=" * 70)
print("TESTES DA CÉLULA 9a CONCLUÍDOS")
print("=" * 70)

TESTES — LIQUIDAÇÃO DE APOSTAS (AUTO-SUFICIENTE)
2026-07-14 21:56:01,378 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-14 21:56:01,386 INFO sqlalchemy.engine.Engine SELECT usuario.id_usuario AS usuario_id_usuario, usuario.nome AS usuario_nome, usuario.email AS usuario_email, usuario.cpf AS usuario_cpf, usuario.data_nascimento AS usuario_data_nascimento, usuario.login AS usuario_login, usuario.senha_hash AS usuario_senha_hash, usuario.status AS usuario_status, usuario.tipo_usuario AS usuario_tipo_usuario 
FROM usuario 
WHERE usuario.login = ?
 LIMIT ? OFFSET ?


INFO:sqlalchemy.engine.Engine:SELECT usuario.id_usuario AS usuario_id_usuario, usuario.nome AS usuario_nome, usuario.email AS usuario_email, usuario.cpf AS usuario_cpf, usuario.data_nascimento AS usuario_data_nascimento, usuario.login AS usuario_login, usuario.senha_hash AS usuario_senha_hash, usuario.status AS usuario_status, usuario.tipo_usuario AS usuario_tipo_usuario 
FROM usuario 
WHERE usuario.login = ?
 LIMIT ? OFFSET ?


2026-07-14 21:56:01,391 INFO sqlalchemy.engine.Engine [cached since 2.352s ago] ('conta_teste_2026', 1, 0)


...

2026-07-14 21:56:02,429 INFO sqlalchemy.engine.Engine [cached since 2.674s ago] (2, 20, 0)


INFO:sqlalchemy.engine.Engine:[cached since 2.674s ago] (2, 20, 0)



Últimas movimentações:
  [CREDITO]    3.4 pts — Aposta ganha — Jogo 6: Coreia do Sul 1x1 República Tcheca | Retorno: 3.40 pts
  [ESTORNO]    1.5 pts — Reembolso — Jogo 5: África do Sul x República Tcheca cancelado | Teste de cancelamento
  [CREDITO]    1.9 pts — Aposta ganha — Jogo 3: México 2x0 República Tcheca | Retorno: 1.95 pts
  [DEBITO ]    1.0 pts — Aposta Jogo 6: Coreia do Sul x República Tcheca | valor 1.0
  [DEBITO ]    1.5 pts — Aposta Jogo 5: África do Sul x República Tcheca | valor 1.5
  [DEBITO ]    2.0 pts — Aposta Jogo 4: África do Sul x Coreia do Sul | valor 2.0
  [DEBITO ]    1.0 pts — Aposta Jogo 3: México x República Tcheca | valor 1.0
  [CREDITO]  100.0 pts — Recarga para testes — Célula 9a
  [DEBITO ]   30.0 pts — Aposta Jogo 2: México x Coreia do Sul | Coreia do Sul vence | Retorno potencial: 55.20
  [DEBITO ]   50.0 pts — Aposta Jogo 1: México x África do Sul | México vence | Retorno potencial: 82.50
  [CREDITO]  200.0 pts — Recarga para testes da Célula 8a
  [

INFO:sqlalchemy.engine.Engine:ROLLBACK



TESTES DA CÉLULA 9a CONCLUÍDOS


In [18]:
# Célula 10 — Funções de Aposta
from sqlalchemy.orm import Session
from datetime import datetime, timezone

# ─────────────────────────────────────────────────────────────
# CORE INTERNO — não comita, opera dentro da sessão recebida
# ─────────────────────────────────────────────────────────────
def _criar_aposta_core(session, id_usuario: int, id_partida: int, palpite: str,
                        pontos: float, multiplicador: int = 1) -> dict:
    """
    Cria uma aposta dentro da sessão fornecida, validando regras de negócio.
    Não comita — quem chamar decide commit/rollback.
    Retorna dict: {sucesso: bool, id_aposta: int, retorno_potencial: float, erro: str}
    """
    # Validações de entrada
    if palpite not in ("selecao_a", "empate", "selecao_b"):
        return {"sucesso": False, "erro": "Palpite inválido."}
    if pontos <= 0:
        return {"sucesso": False, "erro": "Pontos devem ser maiores que zero."}
    if multiplicador not in (1, 2, 3, 4, 5):
        return {"sucesso": False, "erro": "Multiplicador inválido (use 1 a 5)."}

    # Buscar conta do usuário (lock otimista via with_for_update quando suportado)
    conta = (
        session.query(Conta_Pontos)
        .filter_by(id_usuario=id_usuario)
        .with_for_update()
        .first()
    )
    if not conta:
        return {"sucesso": False, "erro": f"Conta do usuário {id_usuario} não encontrada."}

    # Verificar saldo suficiente
    if round(conta.saldo, 2) < round(pontos, 2):
        return {"sucesso": False, "erro": f"Saldo insuficiente. Saldo atual: {conta.saldo:.2f}."}

    # Buscar partida e validar status
    partida = session.get(Partida, id_partida)
    if not partida:
        return {"sucesso": False, "erro": f"Partida {id_partida} não encontrada."}
    if partida.status not in ("agendada", "ao_vivo"):
        return {"sucesso": False, "erro": f"Partida com status '{partida.status}' não aceita apostas."}

    # Determinar odd aplicada conforme o palpite
    if palpite == "selecao_a":
        odd = partida.odd_a
    elif palpite == "selecao_b":
        odd = partida.odd_b
    else:
        odd = partida.odd_empate

    retorno = round(pontos * odd * multiplicador, 2)
    agora = datetime.now(timezone.utc)

    # Criar aposta
    aposta = Aposta(
        id_usuario=id_usuario,
        id_partida=id_partida,
        palpite=palpite,
        pontos_apostados=pontos,
        multiplicador=multiplicador,
        odd_aplicada=odd,
        retorno_potencial=retorno,
        status="ativa",
        data_hora=agora,
    )
    session.add(aposta)
    session.flush()  # garante aposta.id_aposta disponível

    # Debitar saldo e registrar movimentação
    saldo_anterior = conta.saldo
    conta.saldo = round(conta.saldo - pontos, 2)
    conta.data_atualizacao = agora

    nome_a = session.get(Selecao, partida.id_selecao_a).nome
    nome_b = session.get(Selecao, partida.id_selecao_b).nome

    mov = Movimentacao_Pontos(
        id_conta=conta.id_conta,
        id_aposta=aposta.id_aposta,
        tipo_movimentacao="debito",
        pontos=pontos,
        saldo_anterior=saldo_anterior,
        saldo_atual=conta.saldo,
        data_hora=agora,
        descricao=f"Aposta Jogo {partida.num_jogo}: {nome_a} x {nome_b} | valor {pontos} | palpite {palpite}",
    )
    session.add(mov)

    return {
        "sucesso": True,
        "id_aposta": aposta.id_aposta,
        "retorno_potencial": retorno,
        "saldo_apos": conta.saldo,
    }


# ─────────────────────────────────────────────────────────────
# FUNÇÃO PÚBLICA — abre sessão própria e comita
# ─────────────────────────────────────────────────────────────
def criar_aposta(id_usuario: int, id_partida: int, palpite: str,
                  pontos: float, multiplicador: int = 1) -> dict:
    """
    Cria uma aposta com sessão própria (uso isolado, fora de testes em lote).
    Para criar várias apostas na mesma transação, use _criar_aposta_core(session, ...)
    dentro de um bloco `with Session(engine) as session:` já aberto.
    """
    with Session(engine) as session:
        try:
            resultado = _criar_aposta_core(session, id_usuario, id_partida, palpite, pontos, multiplicador)
            if not resultado["sucesso"]:
                session.rollback()
                return resultado
            session.commit()
            return resultado
        except Exception as e:
            session.rollback()
            return {"sucesso": False, "erro": f"Erro ao criar aposta: {str(e)}"}


# ─────────────────────────────────────────────────────────────
# FUNÇÃO AUXILIAR — listar apostas de um usuário
# ─────────────────────────────────────────────────────────────
def listar_apostas_usuario(login: str, limite: int = 20) -> dict:
    """Retorna as apostas mais recentes de um usuário pelo login."""
    with Session(engine) as session:
        usuario = session.query(Usuario).filter_by(login=login).first()
        if not usuario:
            return {"sucesso": False, "erro": "Usuário não encontrado."}

        apostas = (
            session.query(Aposta)
            .filter_by(id_usuario=usuario.id_usuario)
            .order_by(Aposta.data_hora.desc())
            .limit(limite)
            .all()
        )

        resultado = []
        for a in apostas:
            partida = session.get(Partida, a.id_partida)
            nome_a = session.get(Selecao, partida.id_selecao_a).nome
            nome_b = session.get(Selecao, partida.id_selecao_b).nome
            resultado.append({
                "id_aposta": a.id_aposta,
                "jogo": f"{nome_a} x {nome_b}",
                "palpite": a.palpite,
                "pontos_apostados": a.pontos_apostados,
                "multiplicador": a.multiplicador,
                "retorno_potencial": a.retorno_potencial,
                "status": a.status,
            })

        return {"sucesso": True, "total": len(resultado), "apostas": resultado}


print("✅ Célula 10 carregada — funções de aposta disponíveis:")
print("   → criar_aposta(id_usuario, id_partida, palpite, pontos, multiplicador)")
print("   → _criar_aposta_core(session, ...)  [uso interno/testes em lote]")
print("   → listar_apostas_usuario(login, limite)")

✅ Célula 10 carregada — funções de aposta disponíveis:
   → criar_aposta(id_usuario, id_partida, palpite, pontos, multiplicador)
   → _criar_aposta_core(session, ...)  [uso interno/testes em lote]
   → listar_apostas_usuario(login, limite)


In [21]:
# Célula 10a — Teste rápido da Célula 10 consolidada
print("=" * 60)
print("TESTE RÁPIDO — criar_aposta() consolidada")
print("=" * 60)

LOGIN_TESTE = "conta_teste_2026"

with Session(engine) as session:
    usuario = session.query(Usuario).filter_by(login=LOGIN_TESTE).first()
    assert usuario is not None, "Usuário de teste não encontrado"
    id_usuario = usuario.id_usuario

    conta = session.query(Conta_Pontos).filter_by(id_usuario=id_usuario).first()
    saldo_antes = round(conta.saldo, 2)
    print(f"Saldo antes: {saldo_antes:.2f} pts")

    # Buscar uma partida agendada livre (sem aposta do usuário)
    partida_livre = (
        session.query(Partida)
        .filter(
            Partida.status == "agendada",
            ~Partida.id_partida.in_(
                session.query(Aposta.id_partida).filter(Aposta.id_usuario == id_usuario)
            )
        )
        .order_by(Partida.num_jogo.asc())
        .first()
    )
    assert partida_livre is not None, "Nenhuma partida livre encontrada para o teste"
    id_partida_teste = partida_livre.id_partida
    print(f"Partida escolhida: id={id_partida_teste}, num_jogo={partida_livre.num_jogo}")

# Teste 1 — aposta válida
r1 = criar_aposta(id_usuario, id_partida_teste, "selecao_a", 1.0, multiplicador=1)
assert r1["sucesso"], f"Falha inesperada: {r1}"
print(f"\n✅ Teste 1 (aposta válida) OK — id_aposta={r1['id_aposta']}, retorno_potencial={r1['retorno_potencial']}")

# Teste 2 — palpite inválido
r2 = criar_aposta(id_usuario, id_partida_teste, "time_fantasma", 1.0)
assert not r2["sucesso"], "Deveria ter falhado com palpite inválido"
print(f"✅ Teste 2 (palpite inválido) bloqueado corretamente — erro: {r2['erro']}")

# Teste 3 — saldo insuficiente
r3 = criar_aposta(id_usuario, id_partida_teste, "selecao_b", 999999.0)
assert not r3["sucesso"], "Deveria ter falhado por saldo insuficiente"
print(f"✅ Teste 3 (saldo insuficiente) bloqueado corretamente — erro: {r3['erro']}")

# Teste 4 — multiplicador inválido
r4 = criar_aposta(id_usuario, id_partida_teste, "empate", 1.0, multiplicador=9)
assert not r4["sucesso"], "Deveria ter falhado por multiplicador inválido"
print(f"✅ Teste 4 (multiplicador inválido) bloqueado corretamente — erro: {r4['erro']}")

# Teste 5 — listar apostas do usuário
r5 = listar_apostas_usuario(LOGIN_TESTE, limite=5)
assert r5["sucesso"], f"Falha ao listar apostas: {r5}"
print(f"\n✅ Teste 5 (listar apostas) OK — total retornado: {r5['total']}")
for a in r5["apostas"][:3]:
    print(f"   [{a['status'].upper():10}] {a['jogo']} | palpite={a['palpite']} | pontos={a['pontos_apostados']}")

print("\n" + "=" * 60)
print("TODOS OS TESTES DA CÉLULA 10a PASSARAM ✅")
print("=" * 60)

TESTE RÁPIDO — criar_aposta() consolidada
2026-07-14 21:58:45,058 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-14 21:58:45,061 INFO sqlalchemy.engine.Engine SELECT usuario.id_usuario AS usuario_id_usuario, usuario.nome AS usuario_nome, usuario.email AS usuario_email, usuario.cpf AS usuario_cpf, usuario.data_nascimento AS usuario_data_nascimento, usuario.login AS usuario_login, usuario.senha_hash AS usuario_senha_hash, usuario.status AS usuario_status, usuario.tipo_usuario AS usuario_tipo_usuario 
FROM usuario 
WHERE usuario.login = ?
 LIMIT ? OFFSET ?


INFO:sqlalchemy.engine.Engine:SELECT usuario.id_usuario AS usuario_id_usuario, usuario.nome AS usuario_nome, usuario.email AS usuario_email, usuario.cpf AS usuario_cpf, usuario.data_nascimento AS usuario_data_nascimento, usuario.login AS usuario_login, usuario.senha_hash AS usuario_senha_hash, usuario.status AS usuario_status, usuario.tipo_usuario AS usuario_tipo_usuario 
FROM usuario 
WHERE usuario.login = ?
 LIMIT ? OFFSET ?


2026-07-14 21:58:45,064 INFO sqlalchemy.engine.Engine [cached since 166s ago] ('conta_teste_2026', 1, 0)


...

2026-07-14 21:58:45,227 INFO sqlalchemy.engine.Engine [cached since 165.1s ago] (4,)


INFO:sqlalchemy.engine.Engine:[cached since 165.1s ago] (4,)


2026-07-14 21:58:45,231 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK



✅ Teste 5 (listar apostas) OK — total retornado: 5
   [ATIVA     ] Suíça x Canadá | palpite=selecao_a | pontos=1.0
   [GANHA     ] Coreia do Sul x República Tcheca | palpite=empate | pontos=1.0
   [REEMBOLSADA] África do Sul x República Tcheca | palpite=empate | pontos=1.5

TODOS OS TESTES DA CÉLULA 10a PASSARAM ✅


In [22]:
# Célula 12 — VIEW de Ranking + função de consulta
from sqlalchemy import text
from sqlalchemy.orm import Session

# ─────────────────────────────────────────────────────────────
# CRIAÇÃO DA VIEW (SQL nativo — SQLite)
# ─────────────────────────────────────────────────────────────
with engine.connect() as conn:
    conn.execute(text("DROP VIEW IF EXISTS ranking_geral"))
    conn.execute(text("""
        CREATE VIEW ranking_geral AS
        SELECT
            u.id_usuario,
            u.nome,
            u.login,
            cp.saldo,
            COUNT(CASE WHEN a.status = 'ganha'       THEN 1 END) AS apostas_ganhas,
            COUNT(CASE WHEN a.status = 'perdida'     THEN 1 END) AS apostas_perdidas,
            COUNT(CASE WHEN a.status = 'ativa'       THEN 1 END) AS apostas_ativas,
            COUNT(CASE WHEN a.status = 'reembolsada' THEN 1 END) AS apostas_reembolsadas,
            COUNT(a.id_aposta) AS total_apostas
        FROM usuario u
        JOIN conta_pontos cp ON cp.id_usuario = u.id_usuario
        LEFT JOIN aposta a   ON a.id_usuario = u.id_usuario
        WHERE u.tipo_usuario = 'usuario'
        GROUP BY u.id_usuario, u.nome, u.login, cp.saldo
    """))
    conn.commit()

print("✅ VIEW 'ranking_geral' criada com sucesso.")


# ─────────────────────────────────────────────────────────────
# FUNÇÃO — consultar_ranking
# ─────────────────────────────────────────────────────────────
def consultar_ranking(limite: int = 10) -> dict:
    """
    Retorna o ranking geral de usuários ordenado por saldo (desc).
    Inclui estatísticas de apostas e taxa de acerto (%).
    """
    with Session(engine) as session:
        rows = session.execute(
            text("SELECT * FROM ranking_geral ORDER BY saldo DESC LIMIT :limite"),
            {"limite": limite}
        ).mappings().all()

        ranking = []
        for i, row in enumerate(rows, start=1):
            ganhas = row["apostas_ganhas"]
            perdidas = row["apostas_perdidas"]
            decididas = ganhas + perdidas
            taxa_acerto = round((ganhas / decididas) * 100, 1) if decididas > 0 else 0.0

            ranking.append({
                "posicao": i,
                "login": row["login"],
                "nome": row["nome"],
                "saldo": round(row["saldo"], 2),
                "apostas_ganhas": ganhas,
                "apostas_perdidas": perdidas,
                "apostas_ativas": row["apostas_ativas"],
                "apostas_reembolsadas": row["apostas_reembolsadas"],
                "total_apostas": row["total_apostas"],
                "taxa_acerto_pct": taxa_acerto,
            })

        return {"sucesso": True, "total_usuarios": len(ranking), "ranking": ranking}


print("✅ Célula 12 carregada — função disponível:")
print("   → consultar_ranking(limite=10)")

2026-07-14 22:00:48,028 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-14 22:00:48,038 INFO sqlalchemy.engine.Engine DROP VIEW IF EXISTS ranking_geral


INFO:sqlalchemy.engine.Engine:DROP VIEW IF EXISTS ranking_geral


2026-07-14 22:00:48,051 INFO sqlalchemy.engine.Engine [generated in 0.02282s] ()


...

2026-07-14 22:00:48,063 INFO sqlalchemy.engine.Engine [generated in 0.00546s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00546s] ()


2026-07-14 22:00:48,086 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


✅ VIEW 'ranking_geral' criada com sucesso.
✅ Célula 12 carregada — função disponível:
   → consultar_ranking(limite=10)


In [24]:
# Célula 13 — Testes Completos (Ponta a Ponta)
print("=" * 70)
print("TESTES COMPLETOS — INTEGRAÇÃO DE FUNCIONALIDADES")
print("=" * 70)

# ─────────────────────────────────────────────────────────────
# 1. SETUP: CRIAR NOVO USUÁRIO DE TESTE
# ─────────────────────────────────────────────────────────────
print("\n--- 1. Criando novo usuário de teste ---")
novo_login = "teste_completo_user"
novo_email = "teste.completo@email.com"
novo_cpf = "123.456.789-00" # CPF CORRIGIDO para ser válido
novo_senha = "Senha@123"

# Garantir que o usuário não exista (para re-execuções)
with Session(engine) as session:
    usuario_existente = session.query(Usuario).filter_by(login=novo_login).first()
    if usuario_existente:
        # Deletar movimentações de apostas
        for aposta in usuario_existente.apostas:
            for mov in aposta.movimentacoes:
                session.delete(mov)
            session.delete(aposta)
        # Deletar movimentações da conta (bônus, recargas)
        if usuario_existente.conta:
            for mov in usuario_existente.conta.movimentacoes:
                session.delete(mov)
            session.delete(usuario_existente.conta)
        session.delete(usuario_existente)
        session.commit()
        print(f"Usuário '{novo_login}' e dados associados limpos para re-execução.")

r_criar_user = criar_usuario(
    nome="Usuário Teste Completo",
    email=novo_email,
    cpf=novo_cpf,
    data_nascimento=datetime(1990, 1, 1),
    login=novo_login,
    senha=novo_senha,
)
assert r_criar_user["sucesso"], f"Falha ao criar usuário: {r_criar_user['erro']}"
id_novo_usuario = r_criar_user["id_usuario"]
print(f"✅ Usuário '{novo_login}' criado com ID: {id_novo_usuario}")

# ─────────────────────────────────────────────────────────────
# 2. FAZER APOSTAS
# ─────────────────────────────────────────────────────────────
print("\n--- 2. Realizando apostas ---")
with Session(engine) as session:
    # Buscar partidas agendadas (garantir que não sejam as mesmas dos testes anteriores)
    partidas_para_apostar = (
        session.query(Partida)
        .filter(
            Partida.status == "agendada",
            ~Partida.id_partida.in_(
                session.query(Aposta.id_partida).filter(Aposta.id_usuario == id_novo_usuario)
            )
        )
        .order_by(Partida.num_jogo.asc())
        .limit(5) # 2 ganhas, 2 perdidas, 1 reembolsada
        .all()
    )
    assert len(partidas_para_apostar) >= 5, "Não há partidas agendadas suficientes para os testes."

    id_p_ganha1 = partidas_para_apostar[0].id_partida
    id_p_ganha2 = partidas_para_apostar[1].id_partida
    id_p_perdida1 = partidas_para_apostar[2].id_partida
    id_p_perdida2 = partidas_para_apostar[3].id_partida
    id_p_reembolso = partidas_para_apostar[4].id_partida

    # Aposta 1 (ganha)
    r_aposta1 = _criar_aposta_core(session, id_novo_usuario, id_p_ganha1, "selecao_a", 5.0)
    assert r_aposta1["sucesso"], f"Falha aposta 1: {r_aposta1['erro']}"
    id_aposta1 = r_aposta1["id_aposta"]
    print(f"✅ Aposta 1 (id={id_aposta1}) criada para partida {id_p_ganha1} (palpite: selecao_a)")

    # Aposta 2 (ganha)
    r_aposta2 = _criar_aposta_core(session, id_novo_usuario, id_p_ganha2, "empate", 3.0)
    assert r_aposta2["sucesso"], f"Falha aposta 2: {r_aposta2['erro']}"
    id_aposta2 = r_aposta2["id_aposta"]
    print(f"✅ Aposta 2 (id={id_aposta2}) criada para partida {id_p_ganha2} (palpite: empate)")

    # Aposta 3 (perdida)
    r_aposta3 = _criar_aposta_core(session, id_novo_usuario, id_p_perdida1, "selecao_b", 7.0)
    assert r_aposta3["sucesso"], f"Falha aposta 3: {r_aposta3['erro']}"
    id_aposta3 = r_aposta3["id_aposta"]
    print(f"✅ Aposta 3 (id={id_aposta3}) criada para partida {id_p_perdida1} (palpite: selecao_b)")

    # Aposta 4 (perdida)
    r_aposta4 = _criar_aposta_core(session, id_novo_usuario, id_p_perdida2, "selecao_a", 4.0)
    assert r_aposta4["sucesso"], f"Falha aposta 4: {r_aposta4['erro']}"
    id_aposta4 = r_aposta4["id_aposta"]
    print(f"✅ Aposta 4 (id={id_aposta4}) criada para partida {id_p_perdida2} (palpite: selecao_a)")

    # Aposta 5 (reembolsada)
    r_aposta5 = _criar_aposta_core(session, id_novo_usuario, id_p_reembolso, "empate", 6.0)
    assert r_aposta5["sucesso"], f"Falha aposta 5: {r_aposta5['erro']}"
    id_aposta5 = r_aposta5["id_aposta"]
    print(f"✅ Aposta 5 (id={id_aposta5}) criada para partida {id_p_reembolso} (palpite: empate)")

    session.commit() # Commit de todas as apostas
    print("✅ Todas as apostas foram commitadas.")

# ─────────────────────────────────────────────────────────────
# 3. LIQUIDAR PARTIDAS
# ─────────────────────────────────────────────────────────────
print("\n--- 3. Liquidando partidas ---")
# Liquidar partidas ganhas
r_liq_ganha1 = liquidar_partida(id_p_ganha1, gols_a=2, gols_b=0) # Palpite 'selecao_a'
assert r_liq_ganha1["sucesso"], f"Falha liq. ganha 1: {r_liq_ganha1['erro']}"
print(f"✅ Partida {id_p_ganha1} liquidada (ganha): {r_liq_ganha1['placar']}")

r_liq_ganha2 = liquidar_partida(id_p_ganha2, gols_a=1, gols_b=1) # Palpite 'empate'
assert r_liq_ganha2["sucesso"], f"Falha liq. ganha 2: {r_liq_ganha2['erro']}"
print(f"✅ Partida {id_p_ganha2} liquidada (ganha): {r_liq_ganha2['placar']}")

# Liquidar partidas perdidas
r_liq_perdida1 = liquidar_partida(id_p_perdida1, gols_a=1, gols_b=0) # Palpite 'selecao_b'
assert r_liq_perdida1["sucesso"], f"Falha liq. perdida 1: {r_liq_perdida1['erro']}"
print(f"✅ Partida {id_p_perdida1} liquidada (perdida): {r_liq_perdida1['placar']}")

r_liq_perdida2 = liquidar_partida(id_p_perdida2, gols_a=0, gols_b=1) # Palpite 'selecao_a'
assert r_liq_perdida2["sucesso"], f"Falha liq. perdida 2: {r_liq_perdida2['erro']}"
print(f"✅ Partida {id_p_perdida2} liquidada (perdida): {r_liq_perdida2['placar']}")

# Cancelar partida (reembolso)
r_cancelar = cancelar_partida(id_p_reembolso, motivo="Partida adiada")
assert r_cancelar["sucesso"], f"Falha cancelamento: {r_cancelar['erro']}"
print(f"✅ Partida {id_p_reembolso} cancelada (reembolso): {r_cancelar['motivo']}")

# ─────────────────────────────────────────────────────────────
# 4. VERIFICAR SALDO FINAL E MOVIMENTAÇÕES
# ─────────────────────────────────────────────────────────────
print("\n--- 4. Verificando saldo e movimentações ---")
with Session(engine) as session:
    conta_final = session.query(Conta_Pontos).filter_by(id_usuario=id_novo_usuario).first()
    saldo_final = round(conta_final.saldo, 2)

    # Saldo inicial do usuário é 100.0 (bônus de boas-vindas)
    # Apostas: -5.0 -3.0 -7.0 -4.0 -6.0 = -25.0
    # Retornos:
    #   Aposta 1 (id_p_ganha1): 5.0 * odd_a (partida.odd_a)
    #   Aposta 2 (id_p_ganha2): 3.0 * odd_empate (partida.odd_empate)
    # Reembolso: +6.0 (da aposta 5)

    # Calcular o saldo esperado com base nas odds reais
    p_ganha1 = session.get(Partida, id_p_ganha1)
    p_ganha2 = session.get(Partida, id_p_ganha2)

    retorno_aposta1 = round(5.0 * p_ganha1.odd_a, 2)
    retorno_aposta2 = round(3.0 * p_ganha2.odd_empate, 2)

    saldo_esperado = round(100.0 - 25.0 + retorno_aposta1 + retorno_aposta2 + 6.0, 2)

    print(f"Saldo inicial (após bônus): 100.00")
    print(f"Débitos totais das apostas: -25.00")
    print(f"Crédito Aposta 1 (5.0 x {p_ganha1.odd_a}): +{retorno_aposta1:.2f}")
    print(f"Crédito Aposta 2 (3.0 x {p_ganha2.odd_empate}): +{retorno_aposta2:.2f}")
    print(f"Reembolso Aposta 5: +6.00")
    print(f"Saldo esperado: {saldo_esperado:.2f}")
    print(f"Saldo final real: {saldo_final:.2f}")

    assert saldo_final == saldo_esperado, f"Saldo final incorreto! Esperado: {saldo_esperado}, Obtido: {saldo_final}"
    print("✅ Saldo final consistente.")

    # 1 bônus + 5 débitos + 2 créditos (ganhas) + 1 estorno (reembolsada) = 9 movimentações
    movs = listar_movimentacoes(novo_login, limite=10)["historico"]
    assert len(movs) == 9, f"Número de movimentações incorreto. Esperado 9, obtido {len(movs)}"
    print("✅ Número de movimentações consistente.")

# ─────────────────────────────────────────────────────────────
# 5. VERIFICAR RANKING
# ─────────────────────────────────────────────────────────────
print("\n--- 5. Verificando ranking ---")
r_ranking = consultar_ranking(limite=5)
assert r_ranking["sucesso"], f"Falha ao consultar ranking: {r_ranking['erro']}"

usuario_no_ranking = next((u for u in r_ranking["ranking"] if u["login"] == novo_login), None)
assert usuario_no_ranking is not None, f"Usuário '{novo_login}' não encontrado no ranking."

print(f"✅ Usuário '{novo_login}' encontrado no ranking na posição {usuario_no_ranking['posicao']}.")
print(f"   Saldo no ranking: {usuario_no_ranking['saldo']:.2f}")
print(f"   Apostas ganhas: {usuario_no_ranking['apostas_ganhas']}")
print(f"   Apostas perdidas: {usuario_no_ranking['apostas_perdidas']}")
print(f"   Apostas ativas: {usuario_no_ranking['apostas_ativas']}")
print(f"   Apostas reembolsadas: {usuario_no_ranking['apostas_reembolsadas']}")
print(f"   Total apostas: {usuario_no_ranking['total_apostas']}")
print(f"   Taxa de acerto: {usuario_no_ranking['taxa_acerto_pct']:.1f}%")

assert usuario_no_ranking["saldo"] == saldo_final, "Saldo no ranking não bate com saldo final da conta."
assert usuario_no_ranking["apostas_ganhas"] == 2, "Contagem de apostas ganhas incorreta."
assert usuario_no_ranking["apostas_perdidas"] == 2, "Contagem de apostas perdidas incorreta."
assert usuario_no_ranking["apostas_ativas"] == 0, "Contagem de apostas ativas incorreta."
assert usuario_no_ranking["apostas_reembolsadas"] == 1, "Contagem de apostas reembolsadas incorreta."
assert usuario_no_ranking["total_apostas"] == 5, "Contagem total de apostas incorreta."
assert usuario_no_ranking["taxa_acerto_pct"] == round((2 / (2 + 2)) * 100, 1), "Taxa de acerto incorreta."

print("✅ Dados do usuário no ranking consistentes.")

print("\n" + "=" * 70)
print("TODOS OS TESTES COMPLETOS DA CÉLULA 13 PASSARAM ✅")
print("=" * 70)

TESTES COMPLETOS — INTEGRAÇÃO DE FUNCIONALIDADES

--- 1. Criando novo usuário de teste ---
2026-07-14 22:05:43,649 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-07-14 22:05:43,653 INFO sqlalchemy.engine.Engine SELECT usuario.id_usuario AS usuario_id_usuario, usuario.nome AS usuario_nome, usuario.email AS usuario_email, usuario.cpf AS usuario_cpf, usuario.data_nascimento AS usuario_data_nascimento, usuario.login AS usuario_login, usuario.senha_hash AS usuario_senha_hash, usuario.status AS usuario_status, usuario.tipo_usuario AS usuario_tipo_usuario 
FROM usuario 
WHERE usuario.login = ?
 LIMIT ? OFFSET ?


INFO:sqlalchemy.engine.Engine:SELECT usuario.id_usuario AS usuario_id_usuario, usuario.nome AS usuario_nome, usuario.email AS usuario_email, usuario.cpf AS usuario_cpf, usuario.data_nascimento AS usuario_data_nascimento, usuario.login AS usuario_login, usuario.senha_hash AS usuario_senha_hash, usuario.status AS usuario_status, usuario.tipo_usuario AS usuario_tipo_usuario 
FROM usuario 
WHERE usuario.login = ?
 LIMIT ? OFFSET ?


2026-07-14 22:05:43,653 INFO sqlalchemy.engine.Engine [cached since 584.6s ago] ('teste_completo_user', 1, 0)


...

2026-07-14 22:05:44,273 INFO sqlalchemy.engine.Engine [generated in 0.00114s] (5,)


INFO:sqlalchemy.engine.Engine:[generated in 0.00114s] (5,)


2026-07-14 22:05:44,278 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK


✅ Usuário 'teste_completo_user' encontrado no ranking na posição 3.
   Saldo no ranking: 99.45
   Apostas ganhas: 2
   Apostas perdidas: 2
   Apostas ativas: 0
   Apostas reembolsadas: 1
   Total apostas: 5
   Taxa de acerto: 50.0%
✅ Dados do usuário no ranking consistentes.

TODOS OS TESTES COMPLETOS DA CÉLULA 13 PASSARAM ✅
